In [ ]:
!pip install openpyxl

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
dataset_path = r"D:\DermaVision_Module1\dataset\UQ_Longitudinal"

In [ ]:
files = os.listdir(dataset_path)

for file in files:
    print(file)

In [ ]:
participant_df = pd.read_excel(
    os.path.join(dataset_path, "Participant's Metadata.xlsx")
)

general_tile_df = pd.read_excel(
    os.path.join(
        dataset_path,
        "General Tile images with link to dermoscopic images.xlsx"
    )
)

general_dermoscopic_df = pd.read_excel(
    os.path.join(
        dataset_path,
        "General Dermosopic images.xlsx"
    )
)

highrisk_tile_df = pd.read_excel(
    os.path.join(
        dataset_path,
        "HighRisk Tile images with link to dermoscopic images.xlsx"
    )
)

highrisk_dermoscopic_df = pd.read_excel(
    os.path.join(
        dataset_path,
        "HighRisk Dermoscopic images.xlsx"
    )
)

print("All files loaded successfully!")

In [ ]:
datasets = {
    "Participant Metadata": participant_df,
    "General Tile": general_tile_df,
    "General Dermoscopic": general_dermoscopic_df,
    "HighRisk Tile": highrisk_tile_df,
    "HighRisk Dermoscopic": highrisk_dermoscopic_df
}

for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    
    print("Shape:", df.shape)
    print("\nColumns:")
    
    for column in df.columns:
        print("-", column)

In [ ]:
print("Participant Metadata IDs:")
print(participant_df["Participant_Number"].head(10).tolist())

print("\nGeneral Tile Participant_visit:")
print(general_tile_df["Participant_visit"].head(10).tolist())

print("\nHighRisk Tile Participant_visit:")
print(highrisk_tile_df["Participant_visit"].head(10).tolist())

In [ ]:
print("GENERAL TILE - Diagnosis distribution:")
print(general_tile_df["Diagnosis"].value_counts(dropna=False))

print("\n" + "=" * 60)

print("GENERAL DERMOSCOPIC - Diagnosis distribution:")
print(general_dermoscopic_df["Diagnosis"].value_counts(dropna=False))

print("\n" + "=" * 60)

print("HIGH RISK TILE - Diagnosis distribution:")
print(highrisk_tile_df["Diagnosis"].value_counts(dropna=False))

print("\n" + "=" * 60)

print("HIGH RISK DERMOSCOPIC - Diagnosis distribution:")
print(highrisk_dermoscopic_df["Diagnosis"].value_counts(dropna=False))

In [ ]:
print("GENERAL DERMOSCOPIC IDs:")
print(
    general_dermoscopic_df[
        "Dermoscopic_Image_ID*(ParticipantID_LesionID_visitID)"
    ].head(10).tolist()
)

print("\nHIGH RISK DERMOSCOPIC IDs:")
print(
    highrisk_dermoscopic_df[
        "Dermoscopic_Image_ID*(ParticipantID_LesionID_visitID)"
    ].head(10).tolist()
)

In [ ]:
print("General Dermoscopic - Unique image IDs:")
print(
    general_dermoscopic_df[
        "Dermoscopic_Image_ID*(ParticipantID_LesionID_visitID)"
    ].nunique()
)

print("\nGeneral Dermoscopic - Total rows:")
print(len(general_dermoscopic_df))


print("\nHighRisk Dermoscopic - Unique image IDs:")
print(
    highrisk_dermoscopic_df[
        "Dermoscopic_Image_ID*(ParticipantID_LesionID_visitID)"
    ].nunique()
)

print("\nHighRisk Dermoscopic - Total rows:")
print(len(highrisk_dermoscopic_df))

In [ ]:
# Create copies so original datasets remain unchanged

general_df = general_dermoscopic_df.copy()
highrisk_df = highrisk_dermoscopic_df.copy()


# Standardize the ID column name

id_column = "Dermoscopic_Image_ID*(ParticipantID_LesionID_visitID)"

general_df = general_df.rename(
    columns={
        id_column: "Image_ID"
    }
)

highrisk_df = highrisk_df.rename(
    columns={
        id_column: "Image_ID"
    }
)


# Combine both dermoscopic datasets

dermoscopic_df = pd.concat(
    [general_df, highrisk_df],
    ignore_index=True
)

print("Combined dataset shape:", dermoscopic_df.shape)

print("\nFirst 10 Image IDs:")
print(dermoscopic_df["Image_ID"].head(10).tolist())

In [ ]:
# Remove .jpg extension

dermoscopic_df["Image_ID_Clean"] = (
    dermoscopic_df["Image_ID"]
    .str.replace(".jpg", "", regex=False)
)


# Split the ID into Participant, Lesion and Visit

id_parts = dermoscopic_df["Image_ID_Clean"].str.split(
    "_",
    expand=True
)

dermoscopic_df["Participant_ID"] = id_parts[0]

dermoscopic_df["Lesion_ID"] = id_parts[1]

dermoscopic_df["Visit_ID"] = id_parts[2]


print(
    dermoscopic_df[
        [
            "Image_ID",
            "Participant_ID",
            "Lesion_ID",
            "Visit_ID"
        ]
    ].head(10)
)

In [ ]:
print("Total rows:", len(dermoscopic_df))

print(
    "Unique participants:",
    dermoscopic_df["Participant_ID"].nunique()
)

print(
    "Unique participant-lesion combinations:",
    dermoscopic_df[
        ["Participant_ID", "Lesion_ID"]
    ].drop_duplicates().shape[0]
)

print("\nGeneral participant IDs example:")
print(
    dermoscopic_df["Participant_ID"]
    .drop_duplicates()
    .head(10)
    .tolist()
)

print("\nHighRisk participant IDs example:")
print(
    dermoscopic_df["Participant_ID"]
    .drop_duplicates()
    .tail(10)
    .tolist()
)

In [ ]:
metadata_ids = set(participant_df["Participant_Number"])

dermoscopic_ids = set(dermoscopic_df["Participant_ID"])

matched_ids = metadata_ids.intersection(dermoscopic_ids)

print("Participants in metadata:", len(metadata_ids))
print("Participants in dermoscopic data:", len(dermoscopic_ids))
print("Matched participants:", len(matched_ids))

In [ ]:
merged_df = dermoscopic_df.merge(
    participant_df,
    left_on="Participant_ID",
    right_on="Participant_Number",
    how="inner"
)

print("Merged dataset shape:", merged_df.shape)

In [ ]:
print("Merged Dataset Columns:\n")

for i, column in enumerate(merged_df.columns, start=1):
    print(f"{i}. {column}")

In [ ]:
lesion_diagnosis_check = (
    merged_df
    .groupby(["Participant_ID", "Lesion_ID"])["Diagnosis"]
    .nunique()
)

print(
    "Total unique participant-lesion combinations:",
    len(lesion_diagnosis_check)
)

print(
    "Lesions with more than one diagnosis:",
    (lesion_diagnosis_check > 1).sum()
)

In [ ]:
lesion_df = (
    merged_df
    .sort_values(["Participant_ID", "Lesion_ID", "Visit_ID"])
    .drop_duplicates(
        subset=["Participant_ID", "Lesion_ID"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("Lesion-level dataset shape:", lesion_df.shape)
print("Unique participants:", lesion_df["Participant_ID"].nunique())
print(
    "Unique lesions:",
    lesion_df[["Participant_ID", "Lesion_ID"]]
    .drop_duplicates()
    .shape[0]
)

In [ ]:
print("Diagnosis distribution at lesion level:\n")

print(
    lesion_df["Diagnosis"]
    .value_counts(dropna=False)
)

In [ ]:
malignant_diagnoses = [
    "melanoma",
    "basal cell carcinoma",
    "squamous cell carcinoma"
]

print("Potential malignant diagnoses:\n")

print(
    lesion_df[
        lesion_df["Diagnosis"].isin(malignant_diagnoses)
    ]["Diagnosis"].value_counts()
)

print("\nTotal potentially malignant lesions:")

print(
    lesion_df["Diagnosis"]
    .isin(malignant_diagnoses)
    .sum()
)

In [ ]:
malignant_diagnoses = [
    "melanoma",
    "basal cell carcinoma",
    "squamous cell carcinoma"
]

participant_malignant = (
    lesion_df
    .groupby("Participant_ID")["Diagnosis"]
    .apply(lambda x: x.isin(malignant_diagnoses).any())
)

print("Participants with at least one malignant lesion:",
      participant_malignant.sum())

print("Participants without malignant lesions:",
      (~participant_malignant).sum())

print("Total participants:",
      len(participant_malignant))

In [ ]:
print("Unique diagnosis labels:\n")

for diagnosis in sorted(lesion_df["Diagnosis"].dropna().unique()):
    print(diagnosis)

In [ ]:
lesion_df["Diagnosis"] = (
    lesion_df["Diagnosis"]
    .str.strip()
    .str.lower()
)

print("Diagnosis labels standardized.")

print("\nUnique diagnoses:")
for diagnosis in sorted(lesion_df["Diagnosis"].dropna().unique()):
    print(diagnosis)

In [ ]:
missing_values = lesion_df.isnull().sum()

missing_values = missing_values[
    missing_values > 0
].sort_values(ascending=False)

print("Columns with missing values:\n")
print(missing_values)

In [ ]:
print(lesion_df.dtypes)

In [ ]:
columns_to_check = [
    "Age_at_first_visit",
    "Gender",
    "Eye_colour",
    "Hair_colour",
    "Innate_skin_colour",
    "Facultative_skin_colour",
    "Occupational_sun_exposure",
    "Leisure_sun_exposure",
    "Child_sunburn_history",
    "Teen_sunburn_history",
    "Adult_sunburn_history",
    "Participant_cancer_excision",
    "Family_history_of_melanoma",
    "Ancestry"
]

for column in columns_to_check:
    print("\n" + "=" * 60)
    print(column)
    print("=" * 60)

    print(lesion_df[column].value_counts(dropna=False))

In [ ]:
columns_to_check = [
    "Number_of_lesions_with_tile_images",
    "Number_of_lesions_with_dermoscopis_images",
    "Number_of_all_dermoscopis_images",
    "Study_arm",
    "Total_frecklilng_score",
    "Number_of_naevi",
    "Lesion_Simple_Location"
]

for column in columns_to_check:
    print("\n" + "=" * 60)
    print(column)
    print("=" * 60)
    
    print(lesion_df[column].value_counts(dropna=False))

In [ ]:
final_feature_candidates = [
    "Age_at_first_visit",
    "Gender",
    "Eye_colour",
    "Hair_colour",
    "Innate_skin_colour",
    "Facultative_skin_colour",
    "Total_frecklilng_score",
    "Occupational_sun_exposure",
    "Leisure_sun_exposure",
    "Child_sunburn_history",
    "Teen_sunburn_history",
    "Adult_sunburn_history",
    "Participant_cancer_excision",
    "Family_history_of_melanoma",
    "Ancestry",
    "Number_of_naevi",
    "Lesion_Simple_Location",
    "Diagnosis"
]

print(
    lesion_df[final_feature_candidates]
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
participant_cancer_check = (
    lesion_df[
        ["Participant_ID", "Participant_cancer_excision"]
    ]
    .drop_duplicates()
)

print(
    participant_cancer_check[
        "Participant_cancer_excision"
    ].value_counts(dropna=False)
)

In [ ]:
lesion_df["Participant_cancer_excision"] = (
    lesion_df["Participant_cancer_excision"]
    .fillna("Unknown")
)

print(
    lesion_df["Participant_cancer_excision"]
    .value_counts(dropna=False)
)

In [ ]:
columns_with_missing = [
    "Family_history_of_melanoma",
    "Adult_sunburn_history",
    "Child_sunburn_history",
    "Teen_sunburn_history",
    "Leisure_sun_exposure",
    "Occupational_sun_exposure",
    "Ancestry",
    "Hair_colour",
    "Innate_skin_colour",
    "Facultative_skin_colour"
]

lesion_df[columns_with_missing] = (
    lesion_df[columns_with_missing]
    .fillna("Unknown")
)

print("Remaining missing values in selected columns:")

print(
    lesion_df[columns_with_missing]
    .isnull()
    .sum()
)


In [ ]:
final_columns = [
    "Participant_ID",
    "Lesion_ID",

    "Age_at_first_visit",
    "Gender",
    "Eye_colour",
    "Hair_colour",

    "Innate_skin_colour",
    "Facultative_skin_colour",

    "Total_frecklilng_score",

    "Occupational_sun_exposure",
    "Leisure_sun_exposure",

    "Child_sunburn_history",
    "Teen_sunburn_history",
    "Adult_sunburn_history",

    "Participant_cancer_excision",
    "Family_history_of_melanoma",

    "Ancestry",
    "Number_of_naevi",

    "Lesion_Simple_Location",

    "Diagnosis"
]

final_missing_check = (
    lesion_df[final_columns]
    .isnull()
    .sum()
)

print(final_missing_check)

In [ ]:
uq_final_df = (
    lesion_df[final_columns]
    .copy()
    .reset_index(drop=True)
)

print("Final UQ Longitudinal Dataset Shape:", uq_final_df.shape)

print("\nColumns:")
print(uq_final_df.columns.tolist())

print("\nFirst 5 rows:")
print(uq_final_df.head())

In [ ]:
import os

output_path = os.path.join(
    dataset_path,
    "UQ_Longitudinal_Final_Structured.csv"
)

uq_final_df.to_csv(
    output_path,
    index=False
)

print("Dataset saved successfully!")
print("Location:")
print(output_path)

print("\nFinal shape:", uq_final_df.shape)

In [ ]:
#######COMBINING THE THREE DATASETS#########

In [ ]:
import os
import pandas as pd

dataset_path = r"D:\DermaVision_Module1\dataset"

uq_final = pd.read_csv(
    os.path.join(
        dataset_path,
        "UQ_Longitudinal",
        "UQ_Longitudinal_Final_Structured.csv"
    )
)

nhanes_final = pd.read_csv(
    os.path.join(
        dataset_path,
        "NHANES",
        "NHANES_Final_Structured.csv"
    )
)

pad_final = pd.read_csv(
    os.path.join(
        dataset_path,
        "PAD_UFES_20",
        "PAD_UFES_20_Final_Structured.csv"
    )
)

print("UQ columns:")
print(uq_final.columns.tolist())

print("\nNHANES columns:")
print(nhanes_final.columns.tolist())

print("\nPAD-UFES-20 columns:")
print(pad_final.columns.tolist())

In [ ]:
print("UQ:")
display(
    uq_final[
        [
            "Age_at_first_visit",
            "Gender",
            "Innate_skin_colour",
            "Facultative_skin_colour",
            "Participant_cancer_excision",
            "Diagnosis"
        ]
    ].head()
)

print("\nNHANES:")
display(
    nhanes_final[
        [
            "Age",
            "Sex",
            "Skin_Sun_Response",
            "Sunburn_Last_Year",
            "Number_of_Sunburns"
        ]
    ].head()
)

print("\nPAD-UFES-20:")
display(
    pad_final[
        [
            "Age",
            "Sex",
            "Skin_Type",
            "Skin_Cancer_History",
            "Cancer_History",
            "Diagnosis"
        ]
    ].head()
)

In [ ]:
print("UQ Age Groups:")
print(uq_final["Age_at_first_visit"].value_counts())

print("\nNHANES Age range:")
print(nhanes_final["Age"].min(), "to", nhanes_final["Age"].max())

print("\nPAD-UFES-20 Age range:")
print(pad_final["Age"].min(), "to", pad_final["Age"].max())

In [ ]:
def get_age_group(age):
    if pd.isna(age):
        return "Unknown"
    elif age <= 17:
        return "0-17"
    elif age <= 23:
        return "18-23"
    elif age <= 35:
        return "24-35"
    elif age <= 45:
        return "36-45"
    elif age <= 55:
        return "46-55"
    elif age <= 65:
        return "56-65"
    else:
        return "66+"


# UQ already contains age groups
uq_final["Age_Group"] = uq_final["Age_at_first_visit"]


# NHANES exact age → age group
nhanes_final["Age_Group"] = nhanes_final["Age"].apply(get_age_group)


# PAD-UFES-20 exact age → age group
pad_final["Age_Group"] = pad_final["Age"].apply(get_age_group)


print("UQ Age Groups:")
print(uq_final["Age_Group"].value_counts())

print("\nNHANES Age Groups:")
print(nhanes_final["Age_Group"].value_counts())

print("\nPAD-UFES-20 Age Groups:")
print(pad_final["Age_Group"].value_counts())

In [ ]:
# Create one common Sex column in all datasets

uq_final["Sex"] = uq_final["Gender"]

# NHANES already has Sex, but standardize capitalization
nhanes_final["Sex"] = nhanes_final["Sex"].replace({
    "MALE": "Male",
    "FEMALE": "Female"
})

# PAD already has Sex, including Unknown
pad_final["Sex"] = pad_final["Sex"].replace({
    "MALE": "Male",
    "FEMALE": "Female"
})

print("UQ Sex:")
print(uq_final["Sex"].value_counts(dropna=False))

print("\nNHANES Sex:")
print(nhanes_final["Sex"].value_counts(dropna=False))

print("\nPAD-UFES-20 Sex:")
print(pad_final["Sex"].value_counts(dropna=False))

In [ ]:
print("UQ - Innate skin colour:")
print(uq_final["Innate_skin_colour"].value_counts(dropna=False))

print("\nUQ - Facultative skin colour:")
print(uq_final["Facultative_skin_colour"].value_counts(dropna=False))

print("\nNHANES - Skin sun response:")
print(nhanes_final["Skin_Sun_Response"].value_counts(dropna=False).sort_index())

print("\nPAD-UFES-20 - Fitzpatrick skin type:")
print(pad_final["Skin_Type"].value_counts(dropna=False).sort_index())

In [ ]:
print("UQ - Child sunburn history:")
print(uq_final["Child_sunburn_history"].value_counts(dropna=False))

print("\nUQ - Teen sunburn history:")
print(uq_final["Teen_sunburn_history"].value_counts(dropna=False))

print("\nUQ - Adult sunburn history:")
print(uq_final["Adult_sunburn_history"].value_counts(dropna=False))

print("\nNHANES - Sunburn last year:")
print(nhanes_final["Sunburn_Last_Year"].value_counts(dropna=False))

print("\nNHANES - Number of sunburns:")
print(nhanes_final["Number_of_Sunburns"].value_counts(dropna=False))

In [ ]:
print("UQ - Participant cancer excision:")
print(
    uq_final["Participant_cancer_excision"]
    .value_counts(dropna=False)
)

print("\nUQ - Family history of melanoma:")
print(
    uq_final["Family_history_of_melanoma"]
    .value_counts(dropna=False)
)

print("\nPAD - Skin cancer history:")
print(
    pad_final["Skin_Cancer_History"]
    .value_counts(dropna=False)
)

print("\nPAD - Cancer history:")
print(
    pad_final["Cancer_History"]
    .value_counts(dropna=False)
)

In [ ]:
print("UQ Diagnosis:")
print(uq_final["Diagnosis"].value_counts(dropna=False))

print("\nPAD-UFES-20 Diagnosis:")
print(pad_final["Diagnosis"].value_counts(dropna=False))

print("\nUnique UQ diagnoses:")
print(sorted(uq_final["Diagnosis"].dropna().unique()))

print("\nUnique PAD diagnoses:")
print(sorted(pad_final["Diagnosis"].dropna().unique()))

In [ ]:
print("UQ columns not part of the basic common structure:\n")

uq_basic = [
    "Participant_ID",
    "Lesion_ID",
    "Age_at_first_visit",
    "Gender",
    "Sex",
    "Age_Group",
    "Diagnosis"
]

print([col for col in uq_final.columns if col not in uq_basic])


print("\nPAD-UFES-20 columns not part of the basic common structure:\n")

pad_basic = [
    "Participant_ID",
    "Lesion_ID",
    "Age",
    "Sex",
    "Age_Group",
    "Diagnosis",
    "Source_Dataset"
]

print([col for col in pad_final.columns if col not in pad_basic])


print("\nNHANES columns not part of the basic common structure:\n")

nhanes_basic = [
    "Participant_ID",
    "Age",
    "Sex",
    "Age_Group",
    "Source_Dataset"
]

print([col for col in nhanes_final.columns if col not in nhanes_basic])

In [ ]:
# Create copies for master integration

uq_master = uq_final.copy()
nhanes_master = nhanes_final.copy()
pad_master = pad_final.copy()


# --------------------------------------------------
# UQ STANDARDIZATION
# --------------------------------------------------

uq_master = uq_master.rename(columns={
    "Age_at_first_visit": "Age"
})

uq_master["Source_Dataset"] = "UQ_Longitudinal"


# --------------------------------------------------
# NHANES STANDARDIZATION
# --------------------------------------------------

# NHANES already has:
# Participant_ID
# Age
# Sex
# Age_Group
# Source_Dataset


# --------------------------------------------------
# PAD-UFES-20 STANDARDIZATION
# --------------------------------------------------

# PAD already has:
# Participant_ID
# Lesion_ID
# Age
# Sex
# Age_Group
# Diagnosis
# Source_Dataset


print("UQ master shape:", uq_master.shape)
print("NHANES master shape:", nhanes_master.shape)
print("PAD master shape:", pad_master.shape)

print("\nUQ Source Dataset:")
print(uq_master["Source_Dataset"].value_counts())

print("\nNHANES Source Dataset:")
print(nhanes_master["Source_Dataset"].value_counts())

print("\nPAD Source Dataset:")
print(pad_master["Source_Dataset"].value_counts())

In [ ]:
master_dataset = pd.concat(
    [
        uq_master,
        nhanes_master,
        pad_master
    ],
    ignore_index=True,
    sort=False
)

print("Final Master Dataset Shape:")
print(master_dataset.shape)

print("\nRows by Source Dataset:")
print(master_dataset["Source_Dataset"].value_counts())

print("\nTotal number of columns:")
print(len(master_dataset.columns))

print("\nMaster Dataset Columns:")
print(master_dataset.columns.tolist())

In [ ]:
# Remove redundant UQ-specific Gender column
master_dataset = master_dataset.drop(columns=["Gender"])

print("Updated Master Dataset Shape:")
print(master_dataset.shape)

print("\nSex distribution by Source Dataset:")
print(
    master_dataset
    .groupby("Source_Dataset")["Sex"]
    .value_counts(dropna=False)
)

print("\nDuplicate column names:")
print(
    master_dataset.columns[
        master_dataset.columns.duplicated()
    ].tolist()
)

print("\nTotal columns:")
print(len(master_dataset.columns))

In [ ]:
print("MASTER DATASET INTEGRITY CHECK")
print("=" * 50)

print("\n1. Missing values per column:\n")
print(
    master_dataset
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

print("\n2. Diagnosis availability by dataset:\n")
print(
    master_dataset
    .groupby("Source_Dataset")["Diagnosis"]
    .apply(lambda x: x.notna().sum())
)

print("\n3. Missing Participant_ID by dataset:\n")
print(
    master_dataset
    .groupby("Source_Dataset")["Participant_ID"]
    .apply(lambda x: x.isna().sum())
)

print("\n4. Source Dataset values:\n")
print(master_dataset["Source_Dataset"].value_counts())

In [ ]:
master_output_path = os.path.join(
    dataset_path,
    "DermaVision_Master_Integrated_Dataset.csv"
)

master_dataset.to_csv(
    master_output_path,
    index=False
)

print("Master dataset saved successfully!")

print("\nLocation:")
print(master_output_path)

print("\nFinal shape:")
print(master_dataset.shape)

In [ ]:
diagnosis_dataset = master_dataset[
    master_dataset["Diagnosis"].notna()
].copy()

print("Diagnosis Dataset Shape:")
print(diagnosis_dataset.shape)

print("\nRows by Source Dataset:")
print(diagnosis_dataset["Source_Dataset"].value_counts())

print("\nDiagnosis distribution:")
print(diagnosis_dataset["Diagnosis"].value_counts())

In [ ]:
diagnosis_summary = (
    diagnosis_dataset["Diagnosis"]
    .value_counts()
    .reset_index()
)

diagnosis_summary.columns = ["Diagnosis", "Count"]

diagnosis_summary["Percentage"] = (
    diagnosis_summary["Count"]
    / diagnosis_summary["Count"].sum()
    * 100
).round(2)

print(diagnosis_summary.to_string(index=False))

In [ ]:
# Keep diagnosis classes with at least 50 samples

class_counts = diagnosis_dataset["Diagnosis"].value_counts()

selected_classes = class_counts[
    class_counts >= 50
].index.tolist()

model_dataset = diagnosis_dataset[
    diagnosis_dataset["Diagnosis"].isin(selected_classes)
].copy()

print("Selected diagnosis classes:")
print(selected_classes)

print("\nModel Dataset Shape:")
print(model_dataset.shape)

print("\nFinal class distribution:")
print(model_dataset["Diagnosis"].value_counts())

print("\nRows excluded:")
print(len(diagnosis_dataset) - len(model_dataset))

In [ ]:
# Check feature availability separately for each source

feature_availability = pd.DataFrame({
    "UQ_Non_Null": model_dataset[
        model_dataset["Source_Dataset"] == "UQ_Longitudinal"
    ].notna().sum(),

    "PAD_Non_Null": model_dataset[
        model_dataset["Source_Dataset"] == "PAD-UFES-20"
    ].notna().sum()
})

feature_availability["UQ_Missing"] = (
    9388 - feature_availability["UQ_Non_Null"]
)

feature_availability["PAD_Missing"] = (
    1847 - feature_availability["PAD_Non_Null"]
)

print(feature_availability)

In [ ]:
# Features shared across both UQ and PAD datasets

common_features = [
    "Age",
    "Age_Group",
    "Sex"
]

baseline_dataset = model_dataset[
    common_features + ["Diagnosis"]
].copy()

print("Baseline Dataset Shape:")
print(baseline_dataset.shape)

print("\nMissing values:")
print(baseline_dataset.isnull().sum())

print("\nFirst 5 rows:")
print(baseline_dataset.head())

In [ ]:
print("Unique values in Age:")
print(baseline_dataset["Age"].value_counts(dropna=False))

print("\nData types:")
print(baseline_dataset.dtypes)

In [ ]:
baseline_dataset = baseline_dataset.drop(columns=["Age"])

print("Updated Baseline Dataset Shape:")
print(baseline_dataset.shape)

print("\nColumns:")
print(baseline_dataset.columns.tolist())

print("\nAge Group distribution:")
print(baseline_dataset["Age_Group"].value_counts())

print("\nSex distribution:")
print(baseline_dataset["Sex"].value_counts())

print("\nDiagnosis distribution:")
print(baseline_dataset["Diagnosis"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = baseline_dataset.drop(columns=["Diagnosis"])
y = baseline_dataset["Diagnosis"]

# Stratified split: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

In [ ]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

# Encode categorical input features
feature_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_encoded = feature_encoder.fit_transform(X_train)
X_test_encoded = feature_encoder.transform(X_test)

# Encode target diagnosis labels
target_encoder = LabelEncoder()

y_train_encoded = target_encoder.fit_transform(y_train)
y_test_encoded = target_encoder.transform(y_test)

print("Encoded training shape:", X_train_encoded.shape)
print("Encoded testing shape:", X_test_encoded.shape)

print("\nFeature names:")
print(feature_encoder.get_feature_names_out())

print("\nDiagnosis class mapping:")
for index, diagnosis in enumerate(target_encoder.classes_):
    print(f"{index} -> {diagnosis}")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Get class indices
classes = np.unique(y_train_encoded)

# Compute balanced class weights
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_encoded
)

# Create dictionary for model training
class_weights = {
    int(class_index): float(weight)
    for class_index, weight in zip(classes, weights)
}

print("Class weights:\n")

for class_index, weight in class_weights.items():
    diagnosis = target_encoder.inverse_transform([class_index])[0]
    print(
        f"{class_index} -> {diagnosis}: {weight:.2f}"
    )

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


# --------------------------------------------------
# 1. Set random seed for reproducibility
# --------------------------------------------------

torch.manual_seed(42)


# --------------------------------------------------
# 2. Define the ANN / MLP Model
# --------------------------------------------------

class ANNModel(nn.Module):

    def __init__(self, input_size, num_classes):

        super(ANNModel, self).__init__()

        # First hidden layer with ReLU
        self.layer1 = nn.Linear(
            input_size,
            32
        )

        self.relu = nn.ReLU()

        self.dropout1 = nn.Dropout(0.2)


        # Second hidden layer with Sigmoid
        self.layer2 = nn.Linear(
            32,
            16
        )

        self.sigmoid = nn.Sigmoid()

        self.dropout2 = nn.Dropout(0.2)


        # Output layer
        self.output_layer = nn.Linear(
            16,
            num_classes
        )

        self.softmax = nn.Softmax(dim=1)


    def forward(self, x):

        # First hidden layer
        x = self.layer1(x)
        x = self.relu(x)
        x = self.dropout1(x)

        # Second hidden layer
        x = self.layer2(x)
        x = self.sigmoid(x)
        x = self.dropout2(x)

        # Output layer
        x = self.output_layer(x)

        # Softmax activation
        x = self.softmax(x)

        return x


# --------------------------------------------------
# 3. Model Initialization
# --------------------------------------------------

input_size = X_train_encoded.shape[1]
num_classes = len(target_encoder.classes_)

model = ANNModel(
    input_size,
    num_classes
)

print(model)


# --------------------------------------------------
# 4. Define Class Weights
# --------------------------------------------------

class_weight_tensor = torch.tensor(
    [
        class_weights[i]
        for i in range(num_classes)
    ],
    dtype=torch.float32
)


# --------------------------------------------------
# 5. Define Loss Function
# --------------------------------------------------

# Since the model explicitly outputs Softmax probabilities,
# use NLLLoss after taking log probabilities.

class WeightedNLLLoss(nn.Module):

    def __init__(self, weights):

        super(WeightedNLLLoss, self).__init__()

        self.weights = weights

        self.loss_function = nn.NLLLoss(
            weight=self.weights
        )


    def forward(self, predictions, targets):

        # Convert probabilities to log probabilities
        log_predictions = torch.log(
            predictions + 1e-8
        )

        return self.loss_function(
            log_predictions,
            targets
        )


criterion = WeightedNLLLoss(
    class_weight_tensor
)


# --------------------------------------------------
# 6. Adam Optimizer
# --------------------------------------------------

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)


# --------------------------------------------------
# 7. Convert Data to PyTorch Tensors
# --------------------------------------------------

X_train_tensor = torch.tensor(
    X_train_encoded,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_encoded,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train_encoded,
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test_encoded,
    dtype=torch.long
)


print("\nModel setup completed successfully!")

print("Input size:", input_size)
print("Number of classes:", num_classes)

print("\nClass weights:")
print(class_weight_tensor)

In [ ]:
from sklearn.model_selection import train_test_split

# Split the existing training data into
# training set and validation set

X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train_encoded,
    y_train_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_train_encoded
)

# Convert to PyTorch tensors

X_train_final_tensor = torch.tensor(
    X_train_final,
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val,
    dtype=torch.float32
)

y_train_final_tensor = torch.tensor(
    y_train_final,
    dtype=torch.long
)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.long
)

print("Final training set:", X_train_final_tensor.shape)
print("Validation set:", X_val_tensor.shape)
print("Test set:", X_test_tensor.shape)

print("\nValidation class distribution:")
print(pd.Series(y_val).value_counts().sort_index())

In [ ]:
# --------------------------------------------------
# 8. PyTorch ANN Training Loop
# --------------------------------------------------

num_epochs = 100
patience = 10

best_val_loss = float("inf")
patience_counter = 0
best_model_state = None

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []


for epoch in range(num_epochs):

    # ----------------------------------------------
    # TRAINING
    # ----------------------------------------------

    model.train()

    optimizer.zero_grad()

    # Forward pass
    train_outputs = model(X_train_final_tensor)

    # Calculate weighted loss
    train_loss = criterion(
        train_outputs,
        y_train_final_tensor
    )

    # Backpropagation
    train_loss.backward()

    # Update weights using Adam
    optimizer.step()


    # Training accuracy
    _, train_predictions = torch.max(
        train_outputs,
        dim=1
    )

    train_accuracy = (
        train_predictions
        == y_train_final_tensor
    ).float().mean().item()


    # ----------------------------------------------
    # VALIDATION
    # ----------------------------------------------

    model.eval()

    with torch.no_grad():

        val_outputs = model(X_val_tensor)

        val_loss = criterion(
            val_outputs,
            y_val_tensor
        )

        _, val_predictions = torch.max(
            val_outputs,
            dim=1
        )

        val_accuracy = (
            val_predictions
            == y_val_tensor
        ).float().mean().item()


    # ----------------------------------------------
    # STORE METRICS
    # ----------------------------------------------

    train_losses.append(train_loss.item())
    val_losses.append(val_loss.item())

    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)


    # ----------------------------------------------
    # PRINT RESULTS
    # ----------------------------------------------

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {train_loss.item():.4f} | "
        f"Train Acc: {train_accuracy * 100:.2f}% | "
        f"Val Loss: {val_loss.item():.4f} | "
        f"Val Acc: {val_accuracy * 100:.2f}%"
    )


    # ----------------------------------------------
    # EARLY STOPPING
    # ----------------------------------------------

    if val_loss.item() < best_val_loss:

        best_val_loss = val_loss.item()

        patience_counter = 0

        # Save best model state
        best_model_state = {
            key: value.clone()
            for key, value in model.state_dict().items()
        }

    else:

        patience_counter += 1

        if patience_counter >= patience:

            print(
                f"\nEarly stopping triggered "
                f"at epoch {epoch + 1}"
            )

            break


# --------------------------------------------------
# LOAD BEST MODEL
# --------------------------------------------------

if best_model_state is not None:

    model.load_state_dict(
        best_model_state
    )

print("\nTraining completed successfully!")

print(
    f"Best Validation Loss: "
    f"{best_val_loss:.4f}"
)

In [ ]:
# --------------------------------------------------
# FINAL TEST SET EVALUATION
# --------------------------------------------------

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# --------------------------------------------------
# CLASS NAMES
# --------------------------------------------------

class_names = [
    "actinic keratosis",
    "atypical melanocytic proliferation",
    "basal cell carcinoma",
    "benign",
    "melanoma",
    "nevus",
    "seborrheic keratosis",
    "squamous cell carcinoma"
]


# --------------------------------------------------
# MODEL EVALUATION
# --------------------------------------------------

model.eval()

with torch.no_grad():

    test_outputs = model(X_test_tensor)

    _, test_predictions = torch.max(
        test_outputs,
        dim=1
    )


# Convert predictions to NumPy
y_pred = test_predictions.cpu().numpy()

# True labels
y_true = np.array(y_test_encoded)


# --------------------------------------------------
# OVERALL TEST ACCURACY
# --------------------------------------------------

test_accuracy = accuracy_score(
    y_true,
    y_pred
)

print("=" * 60)
print("FINAL TEST SET RESULTS")
print("=" * 60)

print(
    f"\nTest Accuracy: "
    f"{test_accuracy * 100:.2f}%"
)


# --------------------------------------------------
# CLASSIFICATION REPORT
# --------------------------------------------------

print("\nCLASSIFICATION REPORT")
print("-" * 60)

print(
    classification_report(
        y_true,
        y_pred,
        labels=list(range(8)),
        target_names=class_names,
        zero_division=0
    )
)


# --------------------------------------------------
# CONFUSION MATRIX
# --------------------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(8))
)

print("\nCONFUSION MATRIX")
print("-" * 60)

print(cm)


# --------------------------------------------------
# TRUE CLASS DISTRIBUTION
# --------------------------------------------------

print("\nTRUE TEST CLASS DISTRIBUTION")
print("-" * 60)

true_distribution = pd.Series(
    y_true
).value_counts().sort_index()

for class_index in range(8):

    count = true_distribution.get(class_index, 0)

    print(
        f"{class_index} -> "
        f"{class_names[class_index]}: "
        f"{count}"
    )


# --------------------------------------------------
# PREDICTED CLASS DISTRIBUTION
# --------------------------------------------------

print("\nPREDICTED CLASS DISTRIBUTION")
print("-" * 60)

predicted_distribution = pd.Series(
    y_pred
).value_counts().sort_index()

for class_index in range(8):

    count = predicted_distribution.get(class_index, 0)

    print(
        f"{class_index} -> "
        f"{class_names[class_index]}: "
        f"{count}"
    )

In [ ]:
# ============================================================
# NEW / MODIFIED ANN MODEL
#
# MODIFICATION:
# Softmax is NOT applied inside forward() during training.
#
# Reason:
# PyTorch CrossEntropyLoss expects raw logits.
#
# Softmax is still implemented and used separately for
# converting logits into class probabilities.
#
# Required components:
# 1. ReLU
# 2. Sigmoid
# 3. Softmax
# 4. Adam Optimizer (will be defined in the next cell)
# ============================================================

import torch
import torch.nn as nn


class ANNModel(nn.Module):

    def __init__(self, input_size, num_classes):

        super(ANNModel, self).__init__()

        # ----------------------------------------------------
        # FIRST HIDDEN LAYER
        # ----------------------------------------------------

        self.layer1 = nn.Linear(
            input_size,
            32
        )

        # REQUIRED ACTIVATION: ReLU
        self.relu = nn.ReLU()

        self.dropout1 = nn.Dropout(
            p=0.2
        )


        # ----------------------------------------------------
        # SECOND HIDDEN LAYER
        # ----------------------------------------------------

        self.layer2 = nn.Linear(
            32,
            16
        )

        # REQUIRED ACTIVATION: Sigmoid
        self.sigmoid = nn.Sigmoid()

        self.dropout2 = nn.Dropout(
            p=0.2
        )


        # ----------------------------------------------------
        # OUTPUT LAYER
        # ----------------------------------------------------

        self.output_layer = nn.Linear(
            16,
            num_classes
        )


        # ----------------------------------------------------
        # REQUIRED ACTIVATION: Softmax
        #
        # Softmax is defined here but NOT applied inside
        # forward() because CrossEntropyLoss requires raw logits.
        # ----------------------------------------------------

        self.softmax = nn.Softmax(
            dim=1
        )


    def forward(self, x):

        # ----------------------------------------------------
        # INPUT → FIRST HIDDEN LAYER
        # ----------------------------------------------------

        x = self.layer1(x)

        # Apply ReLU
        x = self.relu(x)

        x = self.dropout1(x)


        # ----------------------------------------------------
        # FIRST HIDDEN LAYER → SECOND HIDDEN LAYER
        # ----------------------------------------------------

        x = self.layer2(x)

        # Apply Sigmoid
        x = self.sigmoid(x)

        x = self.dropout2(x)


        # ----------------------------------------------------
        # OUTPUT LAYER
        # ----------------------------------------------------

        # IMPORTANT:
        # Return raw logits for CrossEntropyLoss
        logits = self.output_layer(x)

        return logits


    def predict_proba(self, x):

        # ----------------------------------------------------
        # PREDICTION PROBABILITIES
        #
        # Forward pass gives raw logits
        # Softmax converts them into probabilities
        # ----------------------------------------------------

        logits = self.forward(x)

        probabilities = self.softmax(logits)

        return probabilities


print("New ANNModel class created successfully!")

In [ ]:
# ============================================================
# CELL 2 - MODEL, CLASS WEIGHTS, LOSS FUNCTION AND OPTIMIZER
# MODIFIED: Recreates class_weights_tensor in this cell
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.utils.class_weight import compute_class_weight


# ============================================================
# 1. RECREATE CLASS WEIGHTS
# MODIFIED: This prevents NameError for class_weights_tensor
# ============================================================

# Convert training labels to NumPy array if needed
y_train_array = np.array(y_train)

# Get all class labels
classes = np.unique(y_train_array)

# Calculate balanced class weights
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_array
)

# Convert class weights to PyTorch tensor
class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32
)

print("Class weights recreated successfully:")

for class_index, weight in enumerate(class_weights_tensor):
    print(
        f"{class_index} -> "
        f"{class_names[class_index]}: "
        f"{weight.item():.4f}"
    )


# ============================================================
# 2. DEFINE THE ANN / MLP MODEL
# ============================================================

class ANNModel(nn.Module):

    def __init__(self, input_size, num_classes):
        super(ANNModel, self).__init__()

        # First hidden layer
        self.layer1 = nn.Linear(
            input_size,
            32
        )

        # REQUIRED: ReLU activation
        self.relu = nn.ReLU()

        # Dropout
        self.dropout1 = nn.Dropout(
            p=0.2
        )

        # Second hidden layer
        self.layer2 = nn.Linear(
            32,
            16
        )

        # REQUIRED: Sigmoid activation
        self.sigmoid = nn.Sigmoid()

        # Dropout
        self.dropout2 = nn.Dropout(
            p=0.2
        )

        # Output layer
        self.output_layer = nn.Linear(
            16,
            num_classes
        )

        # REQUIRED: Softmax
        # NOTE:
        # Softmax is defined here to satisfy the ANN requirement.
        # It should NOT be applied before CrossEntropyLoss.
        self.softmax = nn.Softmax(
            dim=1
        )


    def forward(self, x):

        # First layer
        x = self.layer1(x)

        # ReLU activation
        x = self.relu(x)

        # Dropout
        x = self.dropout1(x)

        # Second layer
        x = self.layer2(x)

        # Sigmoid activation
        x = self.sigmoid(x)

        # Dropout
        x = self.dropout2(x)

        # Output layer
        logits = self.output_layer(x)

        # IMPORTANT MODIFICATION:
        # Return RAW LOGITS for CrossEntropyLoss.
        # Do NOT apply softmax here during training.
        return logits


# ============================================================
# 3. CREATE MODEL
# ============================================================

input_size = X_train_encoded.shape[1]
num_classes = len(class_names)

model = ANNModel(
    input_size=input_size,
    num_classes=num_classes
)

print("\nModel created successfully!")

print("\nModel Architecture:")
print(model)


# ============================================================
# 4. LOSS FUNCTION
# ============================================================

# CrossEntropyLoss receives RAW LOGITS from forward()
criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


# ============================================================
# 5. ADAM OPTIMIZER
# REQUIRED: Adam Optimizer
# ============================================================

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)


# ============================================================
# 6. FINAL CONFIRMATION
# ============================================================

print("\n============================================================")
print("MODEL SETUP COMPLETED SUCCESSFULLY")
print("============================================================")

print(f"Input size: {input_size}")
print(f"Number of classes: {num_classes}")

print("\nRequirements implemented:")
print("✓ ReLU")
print("✓ Sigmoid")
print("✓ Softmax defined in model")
print("✓ Adam Optimizer")
print("✓ Class-weighted CrossEntropyLoss")

print("\nClass weights tensor:")
print(class_weights_tensor)

In [ ]:
# ============================================================
# CELL 3 - TRAIN THE ANN / MLP MODEL
# ============================================================

import copy
import torch


# ============================================================
# 1. SET TRAINING PARAMETERS
# ============================================================

num_epochs = 100

best_validation_loss = float("inf")

best_model_state = None


# Lists for storing training history
train_losses = []
validation_losses = []

train_accuracies = []
validation_accuracies = []


# ============================================================
# 2. TRAINING LOOP
# ============================================================

for epoch in range(num_epochs):

    # --------------------------------------------------------
    # TRAINING MODE
    # --------------------------------------------------------

    model.train()

    total_train_loss = 0
    correct_train_predictions = 0
    total_train_samples = 0


    # Forward pass
    train_outputs = model(X_train_tensor)

    # Calculate loss
    train_loss = criterion(
        train_outputs,
        y_train_tensor
    )

    # Backpropagation
    optimizer.zero_grad()

    train_loss.backward()

    # Update weights using Adam Optimizer
    optimizer.step()


    # --------------------------------------------------------
    # TRAINING ACCURACY
    # --------------------------------------------------------

    train_predictions = torch.argmax(
        train_outputs,
        dim=1
    )

    correct_train_predictions = (
        train_predictions == y_train_tensor
    ).sum().item()

    total_train_samples = y_train_tensor.size(0)

    train_accuracy = (
        correct_train_predictions /
        total_train_samples
    ) * 100


    # --------------------------------------------------------
    # VALIDATION MODE
    # --------------------------------------------------------

    model.eval()

    with torch.no_grad():

        validation_outputs = model(
            X_val_tensor
        )

        validation_loss = criterion(
            validation_outputs,
            y_val_tensor
        )

        validation_predictions = torch.argmax(
            validation_outputs,
            dim=1
        )

        correct_validation_predictions = (
            validation_predictions == y_val_tensor
        ).sum().item()

        total_validation_samples = (
            y_val_tensor.size(0)
        )

        validation_accuracy = (
            correct_validation_predictions /
            total_validation_samples
        ) * 100


    # --------------------------------------------------------
    # STORE TRAINING HISTORY
    # --------------------------------------------------------

    train_losses.append(
        train_loss.item()
    )

    validation_losses.append(
        validation_loss.item()
    )

    train_accuracies.append(
        train_accuracy
    )

    validation_accuracies.append(
        validation_accuracy
    )


    # --------------------------------------------------------
    # SAVE BEST MODEL
    # --------------------------------------------------------

    if validation_loss.item() < best_validation_loss:

        best_validation_loss = (
            validation_loss.item()
        )

        best_model_state = copy.deepcopy(
            model.state_dict()
        )


    # --------------------------------------------------------
    # PRINT RESULTS
    # --------------------------------------------------------

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {train_loss.item():.4f} | "
        f"Train Acc: {train_accuracy:.2f}% | "
        f"Val Loss: {validation_loss.item():.4f} | "
        f"Val Acc: {validation_accuracy:.2f}%"
    )


# ============================================================
# 3. LOAD THE BEST MODEL
# ============================================================

model.load_state_dict(
    best_model_state
)


# ============================================================
# 4. TRAINING COMPLETED
# ============================================================

print("\n============================================================")
print("TRAINING COMPLETED SUCCESSFULLY")
print("============================================================")

print(
    f"Best Validation Loss: "
    f"{best_validation_loss:.4f}"
)

In [ ]:
# ============================================================
# CELL 4 - FINAL TEST SET EVALUATION
# ============================================================

import torch
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 1. SET MODEL TO EVALUATION MODE
# ============================================================

model.eval()


# ============================================================
# 2. MAKE PREDICTIONS ON TEST SET
# ============================================================

with torch.no_grad():

    test_outputs = model(
        X_test_tensor
    )

    # Get predicted class index
    test_predictions = torch.argmax(
        test_outputs,
        dim=1
    )


# ============================================================
# 3. CONVERT PYTORCH TENSORS TO NUMPY
# ============================================================

y_true = y_test_tensor.cpu().numpy()

y_pred = test_predictions.cpu().numpy()


# ============================================================
# 4. CALCULATE TEST ACCURACY
# ============================================================

test_accuracy = accuracy_score(
    y_true,
    y_pred
) * 100


# ============================================================
# 5. PRINT FINAL RESULTS
# ============================================================

print("=" * 60)
print("FINAL TEST SET RESULTS")
print("=" * 60)

print(
    f"\nTest Accuracy: "
    f"{test_accuracy:.2f}%"
)


# ============================================================
# 6. CLASSIFICATION REPORT
# ============================================================

print("\nCLASSIFICATION REPORT")
print("-" * 60)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        zero_division=0
    )
)


# ============================================================
# 7. CONFUSION MATRIX
# ============================================================

conf_matrix = confusion_matrix(
    y_true,
    y_pred
)

print("\nCONFUSION MATRIX")
print("-" * 60)

print(conf_matrix)


# ============================================================
# 8. TRUE CLASS DISTRIBUTION
# ============================================================

print("\nTRUE TEST CLASS DISTRIBUTION")
print("-" * 60)

unique_true, true_counts = np.unique(
    y_true,
    return_counts=True
)

for class_index, count in zip(
    unique_true,
    true_counts
):

    print(
        f"{class_index} -> "
        f"{class_names[class_index]}: "
        f"{count}"
    )


# ============================================================
# 9. PREDICTED CLASS DISTRIBUTION
# ============================================================

print("\nPREDICTED CLASS DISTRIBUTION")
print("-" * 60)

unique_pred, predicted_counts = np.unique(
    y_pred,
    return_counts=True
)

for class_index, count in zip(
    unique_pred,
    predicted_counts
):

    print(
        f"{class_index} -> "
        f"{class_names[class_index]}: "
        f"{count}"
    )

In [ ]:
# ============================================================
# STEP 1: FEATURE AVAILABILITY ANALYSIS
# ============================================================
# This cell checks how many non-null values each feature has
# in the current model dataset.
# ============================================================

import pandas as pd

# Display total rows
print("Total rows in model dataset:", model_dataset.shape[0])

# Calculate feature availability
feature_availability = pd.DataFrame({
    "Non_Null_Count": model_dataset.notna().sum(),
    "Missing_Count": model_dataset.isna().sum(),
    "Availability_%": (
        model_dataset.notna().sum() / len(model_dataset) * 100
    ).round(2)
})

# Sort from highest to lowest availability
feature_availability = feature_availability.sort_values(
    by="Availability_%",
    ascending=False
)

print("\nFEATURE AVAILABILITY")
print("=" * 70)

print(feature_availability)

In [ ]:
# ============================================================
# STEP 2: CREATE IMPROVED FEATURE DATASET
# ============================================================
# MODIFIED:
# Added UQ clinical, skin characteristic, sun exposure,
# cancer history, and lesion-related features.
#
# PAD rows are temporarily excluded because these UQ-specific
# features are completely missing for PAD-UFES-20.
# ============================================================

# Select only the UQ dataset
improved_dataset = model_dataset[
    model_dataset["Source_Dataset"] == "UQ_Longitudinal"
].copy()

# ------------------------------------------------------------
# MODIFIED: Selected improved input features
# ------------------------------------------------------------

selected_features = [
    "Age_Group",
    "Sex",
    "Eye_colour",
    "Hair_colour",
    "Innate_skin_colour",
    "Facultative_skin_colour",
    "Total_frecklilng_score",
    "Occupational_sun_exposure",
    "Leisure_sun_exposure",
    "Child_sunburn_history",
    "Teen_sunburn_history",
    "Adult_sunburn_history",
    "Participant_cancer_excision",
    "Family_history_of_melanoma",
    "Ancestry",
    "Number_of_naevi",
    "Lesion_Simple_Location"
]

target_column = "Diagnosis"

# Create final improved dataset
improved_dataset = improved_dataset[
    selected_features + [target_column]
].copy()

# ------------------------------------------------------------
# CHECK DATASET INFORMATION
# ------------------------------------------------------------

print("=" * 60)
print("IMPROVED FEATURE DATASET")
print("=" * 60)

print("\nDataset Shape:")
print(improved_dataset.shape)

print("\nSelected Features:")
for feature in selected_features:
    print("-", feature)

print("\nMissing Values:")
print(improved_dataset.isnull().sum())

print("\nDiagnosis Distribution:")
print(improved_dataset[target_column].value_counts())

print("\nNumber of Classes:")
print(improved_dataset[target_column].nunique())

In [ ]:
# ============================================================
# STEP 3: INSPECT UNIQUE VALUES IN ALL FEATURES
# ============================================================
# MODIFIED:
# We inspect every categorical feature before encoding.
# This helps identify inconsistent values, "Unknown" values,
# and high-cardinality features.
# ============================================================

for column in selected_features:
    
    print("\n" + "=" * 70)
    print("FEATURE:", column)
    print("=" * 70)
    
    print("Number of unique values:", improved_dataset[column].nunique())
    
    print("\nValue counts:")
    print(improved_dataset[column].value_counts(dropna=False))

In [ ]:
# ============================================================
# STEP 4: CLEAN AND ENCODE IMPROVED FEATURES
# ============================================================
# MODIFIED:
# 1. Numerical features are kept as numerical values.
# 2. Categorical features are one-hot encoded.
# 3. "Unknown" values are retained as valid categories.
# 4. Number_of_naevi is NOT one-hot encoded.
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ------------------------------------------------------------
# Separate input features and target
# ------------------------------------------------------------

X_improved = improved_dataset[selected_features].copy()

y_improved = improved_dataset[target_column].copy()


# ------------------------------------------------------------
# MODIFIED: Define numerical and categorical features
# ------------------------------------------------------------

numerical_features = [
    "Total_frecklilng_score",
    "Number_of_naevi"
]

categorical_features = [
    "Age_Group",
    "Sex",
    "Eye_colour",
    "Hair_colour",
    "Innate_skin_colour",
    "Facultative_skin_colour",
    "Occupational_sun_exposure",
    "Leisure_sun_exposure",
    "Child_sunburn_history",
    "Teen_sunburn_history",
    "Adult_sunburn_history",
    "Participant_cancer_excision",
    "Family_history_of_melanoma",
    "Ancestry",
    "Lesion_Simple_Location"
]


# ------------------------------------------------------------
# Create preprocessing pipeline
# ------------------------------------------------------------
# Numerical features:
# StandardScaler -> scales numerical values.
#
# Categorical features:
# OneHotEncoder -> converts categories into binary columns.
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            StandardScaler(),
            numerical_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)


# ------------------------------------------------------------
# Fit and transform the improved dataset
# ------------------------------------------------------------

X_improved_encoded = preprocessor.fit_transform(X_improved)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 60)
print("IMPROVED FEATURE ENCODING COMPLETED")
print("=" * 60)

print("\nOriginal feature shape:")
print(X_improved.shape)

print("\nEncoded feature shape:")
print(X_improved_encoded.shape)

print("\nNumber of numerical features:")
print(len(numerical_features))

print("\nNumber of categorical features:")
print(len(categorical_features))

print("\nTotal encoded input features:")
print(X_improved_encoded.shape[1])

print("\nTarget classes:")
print(sorted(y_improved.unique()))

In [ ]:
# ============================================================
# STEP 5: CREATE TRAIN / VALIDATION / TEST SPLIT
# ============================================================
# MODIFIED:
# Split the ORIGINAL dataset BEFORE fitting the preprocessor.
# This prevents data leakage from scaling/encoding.
#
# Final split:
# Training   = 64%
# Validation = 16%
# Testing    = 20%
# ============================================================

from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# First split:
# 80% temporary training data
# 20% final test data
# ------------------------------------------------------------

X_temp, X_test, y_temp, y_test = train_test_split(
    X_improved,
    y_improved,
    test_size=0.20,
    random_state=42,
    stratify=y_improved
)

# ------------------------------------------------------------
# Second split:
# From the 80%, create:
# 80% training
# 20% validation
#
# Final result:
# 64% training
# 16% validation
# ------------------------------------------------------------

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.20,
    random_state=42,
    stratify=y_temp
)

# ------------------------------------------------------------
# Display split information
# ------------------------------------------------------------

print("=" * 60)
print("TRAIN / VALIDATION / TEST SPLIT COMPLETED")
print("=" * 60)

print("\nTraining features shape:", X_train.shape)
print("Validation features shape:", X_val.shape)
print("Testing features shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nValidation target distribution:")
print(y_val.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

In [ ]:
# ============================================================
# STEP 6: FIT PREPROCESSOR ON TRAINING DATA ONLY
# ============================================================
# MODIFIED:
# The preprocessor is fitted ONLY on X_train.
# Validation and test sets are transformed using the
# already-fitted training preprocessor.
#
# This prevents data leakage.
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ------------------------------------------------------------
# Define preprocessing pipeline
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            StandardScaler(),
            numerical_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

# ------------------------------------------------------------
# MODIFIED: Fit ONLY on training data
# ------------------------------------------------------------

X_train_encoded = preprocessor.fit_transform(X_train)

# ------------------------------------------------------------
# Transform validation and test data
# WITHOUT fitting again
# ------------------------------------------------------------

X_val_encoded = preprocessor.transform(X_val)

X_test_encoded = preprocessor.transform(X_test)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 60)
print("TRAINING-ONLY PREPROCESSING COMPLETED")
print("=" * 60)

print("\nEncoded training shape:", X_train_encoded.shape)
print("Encoded validation shape:", X_val_encoded.shape)
print("Encoded testing shape:", X_test_encoded.shape)

print("\nNumber of final input features:")
print(X_train_encoded.shape[1])

# Safety check
assert X_train_encoded.shape[1] == X_val_encoded.shape[1]
assert X_train_encoded.shape[1] == X_test_encoded.shape[1]

print("\n✓ All datasets have matching feature dimensions.")
print("✓ Preprocessor fitted only on training data.")
print("✓ No preprocessing data leakage.")

In [ ]:
# ============================================================
# CELL: PYTORCH TENSORS + TARGET ENCODING + CLASS WEIGHTS
# MODIFIED: Does not require y_val_encoded or y_test_encoded
# ============================================================

import torch
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight


# ------------------------------------------------------------
# 1. Convert encoded feature data to NumPy arrays
# ------------------------------------------------------------

X_train_np = (
    X_train_encoded.toarray()
    if hasattr(X_train_encoded, "toarray")
    else np.array(X_train_encoded)
)

X_val_np = (
    X_val_encoded.toarray()
    if hasattr(X_val_encoded, "toarray")
    else np.array(X_val_encoded)
)

X_test_np = (
    X_test_encoded.toarray()
    if hasattr(X_test_encoded, "toarray")
    else np.array(X_test_encoded)
)


# ------------------------------------------------------------
# 2. MODIFIED: Encode target labels
# Fit encoder ONLY using training labels
# ------------------------------------------------------------

target_encoder = LabelEncoder()

y_train_np = target_encoder.fit_transform(
    np.array(y_train)
)

y_val_np = target_encoder.transform(
    np.array(y_val)
)

y_test_np = target_encoder.transform(
    np.array(y_test)
)


# ------------------------------------------------------------
# 3. Convert features to PyTorch tensors
# ------------------------------------------------------------

X_train_tensor = torch.tensor(
    X_train_np,
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val_np,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_np,
    dtype=torch.float32
)


# ------------------------------------------------------------
# 4. Convert target labels to PyTorch tensors
# ------------------------------------------------------------

y_train_tensor = torch.tensor(
    y_train_np,
    dtype=torch.long
)

y_val_tensor = torch.tensor(
    y_val_np,
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test_np,
    dtype=torch.long
)


# ------------------------------------------------------------
# 5. Calculate class weights using ONLY training data
# ------------------------------------------------------------

classes = np.unique(y_train_np)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_np
)

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32
)


# ------------------------------------------------------------
# 6. Display results
# ------------------------------------------------------------

print("=" * 65)
print("PYTORCH DATA PREPARATION COMPLETED SUCCESSFULLY")
print("=" * 65)

print("\nFEATURE TENSOR SHAPES")
print("-" * 65)
print("Training:", X_train_tensor.shape)
print("Validation:", X_val_tensor.shape)
print("Testing:", X_test_tensor.shape)

print("\nTARGET TENSOR SHAPES")
print("-" * 65)
print("Training:", y_train_tensor.shape)
print("Validation:", y_val_tensor.shape)
print("Testing:", y_test_tensor.shape)

print("\nMODEL INFORMATION")
print("-" * 65)
print("Number of input features:", X_train_tensor.shape[1])
print("Number of classes:", len(classes))

print("\nTARGET CLASS MAPPING")
print("-" * 65)

for class_index, class_name in enumerate(target_encoder.classes_):
    print(f"{class_index} -> {class_name}")


print("\nCLASS WEIGHTS")
print("-" * 65)

for class_index, weight in enumerate(class_weights):
    print(
        f"{class_index} -> "
        f"{target_encoder.classes_[class_index]}: "
        f"{weight:.4f}"
    )


print("\nClass weights tensor:")
print(class_weights_tensor)

print("\n✓ Feature tensors created successfully.")
print("✓ Target labels encoded successfully.")
print("✓ Class weights calculated from training data only.")
print("✓ Ready for the improved ANN/MLP model.")

In [ ]:
# ============================================================
# CELL: IMPROVED ANN / MLP MODEL
# ============================================================

import torch
import torch.nn as nn


# ------------------------------------------------------------
# 1. Define improved ANN / MLP architecture
# ------------------------------------------------------------

class ImprovedANNModel(nn.Module):

    def __init__(self, input_size, num_classes):

        super(ImprovedANNModel, self).__init__()

        # ----------------------------------------------------
        # INPUT LAYER
        # 82 -> 128
        # ----------------------------------------------------

        self.layer1 = nn.Linear(
            input_size,
            128
        )

        # REQUIRED ACTIVATION: ReLU
        self.relu = nn.ReLU()

        self.dropout1 = nn.Dropout(
            p=0.30
        )


        # ----------------------------------------------------
        # HIDDEN LAYER 1
        # 128 -> 64
        # ----------------------------------------------------

        self.layer2 = nn.Linear(
            128,
            64
        )

        self.dropout2 = nn.Dropout(
            p=0.25
        )


        # ----------------------------------------------------
        # HIDDEN LAYER 2
        # 64 -> 32
        # ----------------------------------------------------

        self.layer3 = nn.Linear(
            64,
            32
        )

        # REQUIRED ACTIVATION: Sigmoid
        self.sigmoid = nn.Sigmoid()

        self.dropout3 = nn.Dropout(
            p=0.20
        )


        # ----------------------------------------------------
        # OUTPUT LAYER
        # 32 -> 8
        # ----------------------------------------------------

        self.output_layer = nn.Linear(
            32,
            num_classes
        )

        # REQUIRED: Softmax
        # IMPORTANT:
        # This is defined to satisfy the project requirement.
        # It will NOT be used inside forward() during training
        # because CrossEntropyLoss requires raw logits.
        # ----------------------------------------------------

        self.softmax = nn.Softmax(
            dim=1
        )


    # --------------------------------------------------------
    # FORWARD PASS
    # --------------------------------------------------------

    def forward(self, x):

        # Input Layer
        x = self.layer1(x)
        x = self.relu(x)
        x = self.dropout1(x)

        # Hidden Layer 1
        x = self.layer2(x)
        x = self.relu(x)
        x = self.dropout2(x)

        # Hidden Layer 2
        x = self.layer3(x)
        x = self.sigmoid(x)
        x = self.dropout3(x)

        # Output Layer
        # Returns RAW LOGITS for CrossEntropyLoss
        logits = self.output_layer(x)

        return logits


# ------------------------------------------------------------
# 2. Model parameters
# ------------------------------------------------------------

input_size = X_train_tensor.shape[1]

num_classes = len(
    target_encoder.classes_
)


# ------------------------------------------------------------
# 3. Create model
# ------------------------------------------------------------

model = ImprovedANNModel(
    input_size=input_size,
    num_classes=num_classes
)


# ------------------------------------------------------------
# 4. Loss Function
# ------------------------------------------------------------
# Using class weights calculated from training data
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


# ------------------------------------------------------------
# 5. REQUIRED OPTIMIZER: Adam
# ------------------------------------------------------------

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=0.0001
)


# ------------------------------------------------------------
# 6. Display model information
# ------------------------------------------------------------

print("=" * 65)
print("IMPROVED ANN / MLP MODEL CREATED SUCCESSFULLY")
print("=" * 65)

print("\nMODEL ARCHITECTURE")
print("-" * 65)

print(model)


print("\nMODEL CONFIGURATION")
print("-" * 65)

print("Input features:", input_size)
print("Number of classes:", num_classes)

print("\nNetwork Structure:")
print(f"{input_size} -> 128 -> 64 -> 32 -> {num_classes}")


print("\nPROJECT REQUIREMENTS")
print("-" * 65)

print("✓ ReLU implemented")
print("✓ Sigmoid implemented")
print("✓ Softmax implemented")
print("✓ Adam Optimizer implemented")


print("\nTRAINING CONFIGURATION")
print("-" * 65)

print("Loss Function: Weighted CrossEntropyLoss")
print("Optimizer: Adam")
print("Learning Rate: 0.001")
print("Weight Decay: 0.0001")


print("\n✓ Model is ready for training.")

In [ ]:
# ============================================================
# CELL: TRAINING LOOP WITH VALIDATION + EARLY STOPPING
# ============================================================

import torch
from torch.utils.data import TensorDataset, DataLoader
import copy


# ------------------------------------------------------------
# 1. Create PyTorch datasets
# ------------------------------------------------------------

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)


# ------------------------------------------------------------
# 2. Create DataLoaders
# ------------------------------------------------------------

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)


# ------------------------------------------------------------
# 3. Training settings
# ------------------------------------------------------------

num_epochs = 150

best_val_loss = float("inf")

best_model_state = None

patience = 20

patience_counter = 0


# ------------------------------------------------------------
# 4. Store training history
# ------------------------------------------------------------

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []


# ------------------------------------------------------------
# 5. Training loop
# ------------------------------------------------------------

for epoch in range(num_epochs):

    # ========================================================
    # TRAINING PHASE
    # ========================================================

    model.train()

    running_train_loss = 0.0

    correct_train = 0
    total_train = 0


    for batch_features, batch_labels in train_loader:

        # ----------------------------------------------------
        # Reset gradients
        # ----------------------------------------------------

        optimizer.zero_grad()


        # ----------------------------------------------------
        # Forward pass
        # Model returns RAW LOGITS
        # ----------------------------------------------------

        outputs = model(
            batch_features
        )


        # ----------------------------------------------------
        # Calculate weighted loss
        # ----------------------------------------------------

        loss = criterion(
            outputs,
            batch_labels
        )


        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------

        loss.backward()

        optimizer.step()


        # ----------------------------------------------------
        # Store training loss
        # ----------------------------------------------------

        running_train_loss += (
            loss.item()
            * batch_features.size(0)
        )


        # ----------------------------------------------------
        # Calculate training accuracy
        # ----------------------------------------------------

        _, predicted = torch.max(
            outputs,
            dim=1
        )

        total_train += batch_labels.size(0)

        correct_train += (
            predicted == batch_labels
        ).sum().item()


    # --------------------------------------------------------
    # Average training loss and accuracy
    # --------------------------------------------------------

    epoch_train_loss = (
        running_train_loss
        / len(train_dataset)
    )

    epoch_train_accuracy = (
        correct_train
        / total_train
        * 100
    )


    # ========================================================
    # VALIDATION PHASE
    # ========================================================

    model.eval()

    running_val_loss = 0.0

    correct_val = 0
    total_val = 0


    with torch.no_grad():

        for batch_features, batch_labels in val_loader:

            # ------------------------------------------------
            # Forward pass
            # ------------------------------------------------

            outputs = model(
                batch_features
            )


            # ------------------------------------------------
            # Calculate validation loss
            # ------------------------------------------------

            loss = criterion(
                outputs,
                batch_labels
            )


            running_val_loss += (
                loss.item()
                * batch_features.size(0)
            )


            # ------------------------------------------------
            # Calculate validation accuracy
            # ------------------------------------------------

            _, predicted = torch.max(
                outputs,
                dim=1
            )

            total_val += batch_labels.size(0)

            correct_val += (
                predicted == batch_labels
            ).sum().item()


    # --------------------------------------------------------
    # Average validation loss and accuracy
    # --------------------------------------------------------

    epoch_val_loss = (
        running_val_loss
        / len(val_dataset)
    )

    epoch_val_accuracy = (
        correct_val
        / total_val
        * 100
    )


    # --------------------------------------------------------
    # Store history
    # --------------------------------------------------------

    train_losses.append(
        epoch_train_loss
    )

    val_losses.append(
        epoch_val_loss
    )

    train_accuracies.append(
        epoch_train_accuracy
    )

    val_accuracies.append(
        epoch_val_accuracy
    )


    # --------------------------------------------------------
    # Display epoch results
    # --------------------------------------------------------

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Train Acc: {epoch_train_accuracy:.2f}% | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {epoch_val_accuracy:.2f}%"
    )


    # ========================================================
    # EARLY STOPPING + SAVE BEST MODEL
    # ========================================================

    if epoch_val_loss < best_val_loss:

        best_val_loss = epoch_val_loss

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        patience_counter = 0

    else:

        patience_counter += 1


    # --------------------------------------------------------
    # Stop if validation loss does not improve
    # --------------------------------------------------------

    if patience_counter >= patience:

        print("\n" + "=" * 65)

        print(
            f"EARLY STOPPING ACTIVATED "
            f"AFTER {epoch + 1} EPOCHS"
        )

        print(
            f"Best Validation Loss: "
            f"{best_val_loss:.4f}"
        )

        print("=" * 65)

        break


# ------------------------------------------------------------
# 6. Restore the best model
# ------------------------------------------------------------

model.load_state_dict(
    best_model_state
)


# ------------------------------------------------------------
# 7. Final confirmation
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("TRAINING COMPLETED SUCCESSFULLY")
print("=" * 65)

print(
    f"\nBest Validation Loss: "
    f"{best_val_loss:.4f}"
)

print(
    f"Total Epochs Completed: "
    f"{len(train_losses)}"
)

print("\n✓ Best model restored successfully.")
print("✓ Training history saved.")
print("✓ Ready for final test evaluation.")

In [ ]:
# ============================================================
# CELL: FINAL TEST SET EVALUATION
# ============================================================

import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ------------------------------------------------------------
# 1. Create test dataset and DataLoader
# ------------------------------------------------------------

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)


# ------------------------------------------------------------
# 2. Set model to evaluation mode
# ------------------------------------------------------------

model.eval()

all_predictions = []
all_true_labels = []

test_running_loss = 0.0


# ------------------------------------------------------------
# 3. Evaluate model on test set
# ------------------------------------------------------------

with torch.no_grad():

    for batch_features, batch_labels in test_loader:

        # Forward pass
        outputs = model(batch_features)

        # Calculate loss
        loss = criterion(
            outputs,
            batch_labels
        )

        test_running_loss += (
            loss.item()
            * batch_features.size(0)
        )

        # Convert logits to probabilities using Softmax
        probabilities = model.softmax(outputs)

        # Get predicted class
        predicted = torch.argmax(
            probabilities,
            dim=1
        )

        # Store predictions
        all_predictions.extend(
            predicted.cpu().numpy()
        )

        # Store true labels
        all_true_labels.extend(
            batch_labels.cpu().numpy()
        )


# ------------------------------------------------------------
# 4. Convert results to NumPy arrays
# ------------------------------------------------------------

y_true = np.array(all_true_labels)

y_pred = np.array(all_predictions)


# ------------------------------------------------------------
# 5. Calculate test metrics
# ------------------------------------------------------------

test_loss = (
    test_running_loss
    / len(test_dataset)
)

test_accuracy = accuracy_score(
    y_true,
    y_pred
)


# ------------------------------------------------------------
# 6. Classification Report
# ------------------------------------------------------------

class_names = target_encoder.classes_

report = classification_report(
    y_true,
    y_pred,
    labels=np.arange(num_classes),
    target_names=class_names,
    zero_division=0
)


# ------------------------------------------------------------
# 7. Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(num_classes)
)


# ------------------------------------------------------------
# 8. Display final results
# ------------------------------------------------------------

print("=" * 65)
print("FINAL TEST SET RESULTS")
print("=" * 65)

print(
    f"\nTest Loss: {test_loss:.4f}"
)

print(
    f"Test Accuracy: "
    f"{test_accuracy * 100:.2f}%"
)


print("\nCLASSIFICATION REPORT")
print("-" * 65)

print(report)


print("\nCONFUSION MATRIX")
print("-" * 65)

print(cm)


# ------------------------------------------------------------
# 9. True class distribution
# ------------------------------------------------------------

print("\nTRUE TEST CLASS DISTRIBUTION")
print("-" * 65)

true_classes, true_counts = np.unique(
    y_true,
    return_counts=True
)

for class_index, count in zip(
    true_classes,
    true_counts
):

    print(
        f"{class_index} -> "
        f"{class_names[class_index]}: "
        f"{count}"
    )


# ------------------------------------------------------------
# 10. Predicted class distribution
# ------------------------------------------------------------

print("\nPREDICTED CLASS DISTRIBUTION")
print("-" * 65)

predicted_classes, predicted_counts = np.unique(
    y_pred,
    return_counts=True
)

for class_index, count in zip(
    predicted_classes,
    predicted_counts
):

    print(
        f"{class_index} -> "
        f"{class_names[class_index]}: "
        f"{count}"
    )


print("\n" + "=" * 65)
print("TEST EVALUATION COMPLETED SUCCESSFULLY")
print("=" * 65)

In [ ]:
# ============================================================
# IMPROVED ANN TRAINING PIPELINE - NEW CELL
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler


# ============================================================
# 1. DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)


# ============================================================
# 2. CREATE DATASETS
# ============================================================

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)


# ============================================================
# ================= MODIFIED =================
# WEIGHTED RANDOM SAMPLER
#
# This gives minority classes a higher chance of appearing
# during training.
# ============================================================

class_counts = torch.bincount(y_train_tensor)

print("\nTraining Class Counts:")
for i, count in enumerate(class_counts):
    print(f"{i} -> {class_names[i]}: {count.item()}")


# Inverse frequency for each class
class_sample_weights = 1.0 / class_counts.float()


# Assign each training sample its corresponding class weight
sample_weights = class_sample_weights[y_train_tensor]


# Create weighted sampler
weighted_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)


# ============================================================
# ================= MODIFIED =================
# DATA LOADERS
# ============================================================

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=weighted_sampler
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)


print("\nDataLoaders created successfully!")
print("Batch size:", batch_size)


# ============================================================
# ================= MODIFIED =================
# 3. IMPROVED ANN MODEL
#
# IMPORTANT:
# Softmax is NOT used in forward().
# CrossEntropyLoss expects RAW LOGITS.
# ============================================================

class BetterANNModel(nn.Module):

    def __init__(self, input_size, num_classes):

        super(BetterANNModel, self).__init__()


        # -------------------------------
        # First Hidden Layer
        # -------------------------------

        self.layer1 = nn.Linear(
            input_size,
            256
        )

        self.bn1 = nn.BatchNorm1d(256)

        self.relu1 = nn.ReLU()

        self.dropout1 = nn.Dropout(0.30)


        # -------------------------------
        # Second Hidden Layer
        # -------------------------------

        self.layer2 = nn.Linear(
            256,
            128
        )

        self.bn2 = nn.BatchNorm1d(128)

        self.relu2 = nn.ReLU()

        self.dropout2 = nn.Dropout(0.25)


        # -------------------------------
        # Third Hidden Layer
        # -------------------------------

        self.layer3 = nn.Linear(
            128,
            64
        )

        self.bn3 = nn.BatchNorm1d(64)

        self.relu3 = nn.ReLU()

        self.dropout3 = nn.Dropout(0.20)


        # -------------------------------
        # Fourth Hidden Layer
        # ================= MODIFIED =================
        # Sigmoid retained to satisfy
        # your project requirement.
        # -------------------------------

        self.layer4 = nn.Linear(
            64,
            32
        )

        self.sigmoid = nn.Sigmoid()

        self.dropout4 = nn.Dropout(0.15)


        # -------------------------------
        # Output Layer
        # -------------------------------

        self.output_layer = nn.Linear(
            32,
            num_classes
        )


        # ====================================================
        # Softmax defined for project requirement
        # BUT NOT USED inside forward() during training
        # ====================================================

        self.softmax = nn.Softmax(dim=1)


    def forward(self, x):

        # Layer 1
        x = self.layer1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout1(x)


        # Layer 2
        x = self.layer2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.dropout2(x)


        # Layer 3
        x = self.layer3(x)
        x = self.bn3(x)
        x = self.relu3(x)
        x = self.dropout3(x)


        # Layer 4
        x = self.layer4(x)
        x = self.sigmoid(x)
        x = self.dropout4(x)


        # ====================================================
        # RAW LOGITS OUTPUT
        # DO NOT APPLY SOFTMAX HERE
        # ====================================================

        logits = self.output_layer(x)

        return logits


# ============================================================
# 4. CREATE MODEL
# ============================================================

input_size = X_train_tensor.shape[1]

num_classes = len(class_names)

model = BetterANNModel(
    input_size=input_size,
    num_classes=num_classes
)

model = model.to(device)


print("\n============================================================")
print("IMPROVED MODEL CREATED SUCCESSFULLY")
print("============================================================")

print(model)


# ============================================================
# ================= MODIFIED =================
# 5. MODERATE CLASS WEIGHTS
#
# Previous weights were extremely high, for example 62+.
# We use square root inverse frequency to reduce instability.
# ============================================================

total_samples = len(y_train_tensor)

num_classes = len(class_counts)


moderate_class_weights = torch.sqrt(
    total_samples /
    (
        num_classes * class_counts.float()
    )
)


# Normalize weights around 1
moderate_class_weights = (
    moderate_class_weights /
    moderate_class_weights.mean()
)


moderate_class_weights = moderate_class_weights.to(device)


print("\n============================================================")
print("MODERATE CLASS WEIGHTS")
print("============================================================")

for i, weight in enumerate(moderate_class_weights):
    print(
        f"{i} -> {class_names[i]}: "
        f"{weight.item():.4f}"
    )


# ============================================================
# 6. LOSS FUNCTION
# ============================================================

criterion = nn.CrossEntropyLoss(
    weight=moderate_class_weights
)


# ============================================================
# ================= MODIFIED =================
# 7. ADAMW OPTIMIZER
# ============================================================

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.0005,
    weight_decay=0.0001
)


# ============================================================
# ================= MODIFIED =================
# 8. LEARNING RATE SCHEDULER
# ============================================================

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5
)


# ============================================================
# 9. TRAINING SETTINGS
# ============================================================

num_epochs = 150

early_stopping_patience = 20

best_val_loss = float("inf")

best_model_state = None

epochs_without_improvement = 0


train_losses = []
train_accuracies = []

val_losses = []
val_accuracies = []


print("\n============================================================")
print("TRAINING STARTED")
print("============================================================")


# ============================================================
# 10. TRAINING LOOP
# ============================================================

for epoch in range(num_epochs):


    # ========================================================
    # TRAINING MODE
    # ========================================================

    model.train()


    running_train_loss = 0.0

    correct_train = 0

    total_train = 0


    for batch_X, batch_y in train_loader:


        # Move batch to device
        batch_X = batch_X.to(device)

        batch_y = batch_y.to(device)


        # Clear previous gradients
        optimizer.zero_grad()


        # Forward pass
        outputs = model(batch_X)


        # Calculate loss
        loss = criterion(
            outputs,
            batch_y
        )


        # Backpropagation
        loss.backward()


        # Gradient clipping
        # ================= MODIFIED =================
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )


        # Update model
        optimizer.step()


        # Store loss
        running_train_loss += (
            loss.item() *
            batch_X.size(0)
        )


        # Predictions
        _, predicted = torch.max(
            outputs,
            1
        )


        # Accuracy calculation
        correct_train += (
            predicted == batch_y
        ).sum().item()


        total_train += batch_y.size(0)


    # Calculate training metrics
    train_loss = (
        running_train_loss /
        total_train
    )


    train_accuracy = (
        correct_train /
        total_train
    ) * 100


    # ========================================================
    # VALIDATION MODE
    # ========================================================

    model.eval()


    running_val_loss = 0.0

    correct_val = 0

    total_val = 0


    with torch.no_grad():

        for batch_X, batch_y in val_loader:


            # Move batch to device
            batch_X = batch_X.to(device)

            batch_y = batch_y.to(device)


            # Forward pass
            outputs = model(batch_X)


            # Validation loss
            loss = criterion(
                outputs,
                batch_y
            )


            running_val_loss += (
                loss.item() *
                batch_X.size(0)
            )


            # Predictions
            _, predicted = torch.max(
                outputs,
                1
            )


            # Validation accuracy
            correct_val += (
                predicted == batch_y
            ).sum().item()


            total_val += batch_y.size(0)


    # Calculate validation metrics
    val_loss = (
        running_val_loss /
        total_val
    )


    val_accuracy = (
        correct_val /
        total_val
    ) * 100


    # ========================================================
    # SAVE HISTORY
    # ========================================================

    train_losses.append(train_loss)

    train_accuracies.append(train_accuracy)

    val_losses.append(val_loss)

    val_accuracies.append(val_accuracy)


    # ========================================================
    # UPDATE LEARNING RATE
    # ========================================================

    scheduler.step(val_loss)


    current_lr = (
        optimizer.param_groups[0]["lr"]
    )


    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.2f}% | "
        f"LR: {current_lr:.6f}"
    )


    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if val_loss < best_val_loss:


        best_val_loss = val_loss


        best_model_state = (
            model.state_dict()
        )


        epochs_without_improvement = 0


    else:


        epochs_without_improvement += 1


        if (
            epochs_without_improvement
            >= early_stopping_patience
        ):


            print(
                "\n============================================================"
            )

            print(
                f"EARLY STOPPING ACTIVATED "
                f"AFTER {epoch + 1} EPOCHS"
            )

            print(
                f"Best Validation Loss: "
                f"{best_val_loss:.4f}"
            )

            print(
                "============================================================"
            )

            break


# ============================================================
# 11. RESTORE BEST MODEL
# ============================================================

if best_model_state is not None:

    model.load_state_dict(
        best_model_state
    )


print("\n============================================================")
print("IMPROVED TRAINING COMPLETED SUCCESSFULLY")
print("============================================================")

print(
    f"Best Validation Loss: "
    f"{best_val_loss:.4f}"
)

print(
    f"Total Epochs Completed: "
    f"{len(train_losses)}"
)

print("\n✓ Best model restored successfully.")
print("✓ Weighted sampling was used.")
print("✓ Moderate class weights were used.")
print("✓ Learning rate scheduler was used.")
print("✓ Gradient clipping was used.")
print("✓ Ready for final test evaluation.")

In [ ]:
# ============================================================
# IMPROVED TRAINING PIPELINE - VERSION 2
# ============================================================
# MODIFICATION:
# 1. WeightedRandomSampler is used for class balancing
# 2. CrossEntropyLoss has NO class weights
# 3. Smaller ANN to reduce overfitting
# 4. Raw logits are used with CrossEntropyLoss
# 5. Early stopping + LR scheduler
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    TensorDataset,
    DataLoader,
    WeightedRandomSampler
)


# ============================================================
# 1. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# ============================================================
# 2. CREATE DATASETS
# ============================================================

train_dataset_v2 = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset_v2 = TensorDataset(
    X_val_tensor,
    y_val_tensor
)


# ============================================================
# ================= MODIFIED =================
# 3. WEIGHTED RANDOM SAMPLER ONLY
# ============================================================

class_counts_v2 = torch.bincount(
    y_train_tensor
)

print("\nTraining Class Counts:")

for i, count in enumerate(class_counts_v2):

    print(
        f"{i} -> {class_names[i]}: "
        f"{count.item()}"
    )


# Inverse frequency
class_sample_weights_v2 = (
    1.0 /
    class_counts_v2.float()
)


# Assign weight to every sample
sample_weights_v2 = (
    class_sample_weights_v2[
        y_train_tensor
    ]
)


weighted_sampler_v2 = WeightedRandomSampler(

    weights=sample_weights_v2,

    num_samples=len(sample_weights_v2),

    replacement=True
)


# ============================================================
# 4. DATA LOADERS
# ============================================================

batch_size_v2 = 64


train_loader_v2 = DataLoader(

    train_dataset_v2,

    batch_size=batch_size_v2,

    sampler=weighted_sampler_v2
)


val_loader_v2 = DataLoader(

    val_dataset_v2,

    batch_size=batch_size_v2,

    shuffle=False
)


print("\nDataLoaders created successfully!")
print("Batch size:", batch_size_v2)


# ============================================================
# ================= MODIFIED =================
# 5. SMALLER AND MORE STABLE ANN MODEL
# ============================================================

class BalancedANNModel(nn.Module):


    def __init__(
        self,
        input_size,
        num_classes
    ):

        super(
            BalancedANNModel,
            self
        ).__init__()


        # ----------------------------------------------------
        # Hidden Layer 1
        # ----------------------------------------------------

        self.layer1 = nn.Linear(
            input_size,
            128
        )

        self.bn1 = nn.BatchNorm1d(128)

        self.relu1 = nn.ReLU()

        self.dropout1 = nn.Dropout(0.25)


        # ----------------------------------------------------
        # Hidden Layer 2
        # ----------------------------------------------------

        self.layer2 = nn.Linear(
            128,
            64
        )

        self.bn2 = nn.BatchNorm1d(64)

        self.relu2 = nn.ReLU()

        self.dropout2 = nn.Dropout(0.20)


        # ----------------------------------------------------
        # Hidden Layer 3
        # ----------------------------------------------------
        # Sigmoid retained for project requirement

        self.layer3 = nn.Linear(
            64,
            32
        )

        self.sigmoid = nn.Sigmoid()

        self.dropout3 = nn.Dropout(0.15)


        # ----------------------------------------------------
        # Output Layer
        # ----------------------------------------------------

        self.output_layer = nn.Linear(
            32,
            num_classes
        )


        # ----------------------------------------------------
        # Softmax defined for project requirement
        # IMPORTANT:
        # NOT used in forward()
        # ----------------------------------------------------

        self.softmax = nn.Softmax(
            dim=1
        )


    def forward(self, x):


        # Layer 1

        x = self.layer1(x)

        x = self.bn1(x)

        x = self.relu1(x)

        x = self.dropout1(x)


        # Layer 2

        x = self.layer2(x)

        x = self.bn2(x)

        x = self.relu2(x)

        x = self.dropout2(x)


        # Layer 3

        x = self.layer3(x)

        x = self.sigmoid(x)

        x = self.dropout3(x)


        # ====================================================
        # RAW LOGITS
        # ====================================================

        logits = self.output_layer(x)


        return logits


# ============================================================
# 6. CREATE MODEL
# ============================================================

input_size_v2 = (
    X_train_tensor.shape[1]
)

num_classes_v2 = (
    len(class_names)
)


model_v2 = BalancedANNModel(

    input_size=input_size_v2,

    num_classes=num_classes_v2
)


model_v2 = model_v2.to(device)


print("\n============================================================")
print("BALANCED ANN MODEL CREATED SUCCESSFULLY")
print("============================================================")

print(model_v2)


# ============================================================
# ================= MODIFIED =================
# 7. LOSS FUNCTION
#
# NO CLASS WEIGHTS HERE
# WeightedRandomSampler already handles balancing.
# ============================================================

criterion_v2 = nn.CrossEntropyLoss()


# ============================================================
# ================= MODIFIED =================
# 8. OPTIMIZER
# ============================================================

optimizer_v2 = optim.AdamW(

    model_v2.parameters(),

    lr=0.0003,

    weight_decay=0.0001
)


# ============================================================
# 9. LEARNING RATE SCHEDULER
# ============================================================

scheduler_v2 = optim.lr_scheduler.ReduceLROnPlateau(

    optimizer_v2,

    mode="min",

    factor=0.5,

    patience=8
)


# ============================================================
# 10. TRAINING SETTINGS
# ============================================================

num_epochs_v2 = 150


early_stopping_patience_v2 = 25


best_val_loss_v2 = float("inf")


best_model_state_v2 = None


epochs_without_improvement_v2 = 0


# Training history

train_losses_v2 = []

train_accuracies_v2 = []

val_losses_v2 = []

val_accuracies_v2 = []


print("\n============================================================")
print("VERSION 2 TRAINING STARTED")
print("============================================================")


# ============================================================
# 11. TRAINING LOOP
# ============================================================

for epoch in range(num_epochs_v2):


    # ========================================================
    # TRAINING MODE
    # ========================================================

    model_v2.train()


    running_train_loss_v2 = 0.0


    correct_train_v2 = 0


    total_train_v2 = 0


    for batch_X, batch_y in train_loader_v2:


        # Move data to device

        batch_X = batch_X.to(device)

        batch_y = batch_y.to(device)


        # Clear gradients

        optimizer_v2.zero_grad()


        # Forward pass

        outputs = model_v2(
            batch_X
        )


        # Calculate loss

        loss = criterion_v2(

            outputs,

            batch_y
        )


        # Backpropagation

        loss.backward()


        # Gradient clipping

        torch.nn.utils.clip_grad_norm_(

            model_v2.parameters(),

            max_norm=1.0
        )


        # Update weights

        optimizer_v2.step()


        # Store batch loss

        running_train_loss_v2 += (

            loss.item() *

            batch_X.size(0)
        )


        # Get predictions

        _, predicted = torch.max(

            outputs,

            1
        )


        # Count correct predictions

        correct_train_v2 += (

            predicted == batch_y

        ).sum().item()


        total_train_v2 += (

            batch_y.size(0)
        )


    # Calculate training metrics

    train_loss_v2 = (

        running_train_loss_v2 /

        total_train_v2
    )


    train_accuracy_v2 = (

        correct_train_v2 /

        total_train_v2

    ) * 100


    # ========================================================
    # VALIDATION MODE
    # ========================================================

    model_v2.eval()


    running_val_loss_v2 = 0.0


    correct_val_v2 = 0


    total_val_v2 = 0


    with torch.no_grad():


        for batch_X, batch_y in val_loader_v2:


            batch_X = batch_X.to(
                device
            )

            batch_y = batch_y.to(
                device
            )


            # Forward pass

            outputs = model_v2(
                batch_X
            )


            # Validation loss

            loss = criterion_v2(

                outputs,

                batch_y
            )


            running_val_loss_v2 += (

                loss.item() *

                batch_X.size(0)
            )


            # Predictions

            _, predicted = torch.max(

                outputs,

                1
            )


            correct_val_v2 += (

                predicted == batch_y

            ).sum().item()


            total_val_v2 += (

                batch_y.size(0)
            )


    # ========================================================
    # CALCULATE VALIDATION METRICS
    # ========================================================

    val_loss_v2 = (

        running_val_loss_v2 /

        total_val_v2
    )


    val_accuracy_v2 = (

        correct_val_v2 /

        total_val_v2

    ) * 100


    # ========================================================
    # SAVE TRAINING HISTORY
    # ========================================================

    train_losses_v2.append(
        train_loss_v2
    )

    train_accuracies_v2.append(
        train_accuracy_v2
    )

    val_losses_v2.append(
        val_loss_v2
    )

    val_accuracies_v2.append(
        val_accuracy_v2
    )


    # ========================================================
    # UPDATE LEARNING RATE
    # ========================================================

    scheduler_v2.step(
        val_loss_v2
    )


    current_lr_v2 = (

        optimizer_v2.param_groups[0]["lr"]
    )


    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print(

        f"Epoch [{epoch + 1}/{num_epochs_v2}] | "

        f"Train Loss: {train_loss_v2:.4f} | "

        f"Train Acc: {train_accuracy_v2:.2f}% | "

        f"Val Loss: {val_loss_v2:.4f} | "

        f"Val Acc: {val_accuracy_v2:.2f}% | "

        f"LR: {current_lr_v2:.6f}"
    )


    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if val_loss_v2 < best_val_loss_v2:


        best_val_loss_v2 = (
            val_loss_v2
        )


        best_model_state_v2 = (

            model_v2.state_dict()
        )


        epochs_without_improvement_v2 = 0


    else:


        epochs_without_improvement_v2 += 1


        if (

            epochs_without_improvement_v2

            >= early_stopping_patience_v2
        ):


            print(
                "\n============================================================"
            )

            print(
                f"EARLY STOPPING ACTIVATED AFTER "
                f"{epoch + 1} EPOCHS"
            )

            print(
                f"Best Validation Loss: "
                f"{best_val_loss_v2:.4f}"
            )

            print(
                "============================================================"
            )


            break


# ============================================================
# 12. RESTORE BEST MODEL
# ============================================================

if best_model_state_v2 is not None:


    model_v2.load_state_dict(

        best_model_state_v2
    )


print("\n============================================================")
print("VERSION 2 TRAINING COMPLETED SUCCESSFULLY")
print("============================================================")

print(
    f"Best Validation Loss: "
    f"{best_val_loss_v2:.4f}"
)

print(
    f"Total Epochs Completed: "
    f"{len(train_losses_v2)}"
)

print(
    "\n✓ Best model restored successfully."
)

print(
    "✓ WeightedRandomSampler used for class balancing."
)

print(
    "✓ No class weights used in CrossEntropyLoss."
)

print(
    "✓ Learning rate scheduler used."
)

print(
    "✓ Gradient clipping used."
)

print(
    "✓ Ready for final test evaluation."
)

In [ ]:
# ============================================================
# FINAL TEST SET EVALUATION - NEW CELL
# ============================================================

import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# ------------------------------------------------------------
# 1. Set model to evaluation mode
# ------------------------------------------------------------

model.eval()

# ------------------------------------------------------------
# 2. Variables to store predictions and true labels
# ------------------------------------------------------------

all_predictions = []
all_true_labels = []

test_loss = 0.0
total_correct = 0
total_samples = 0

# ------------------------------------------------------------
# 3. Disable gradient calculation during testing
# ------------------------------------------------------------

with torch.no_grad():

    # Go through all batches in the test DataLoader
    for features, labels in test_loader:

        # Move data to the same device as the model
        features = features.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(features)

        # ----------------------------------------------------
        # IMPORTANT:
        # If your model's forward() currently returns
        # Softmax probabilities, convert safely for evaluation.
        # ----------------------------------------------------

        # Calculate predicted class
        _, predicted = torch.max(outputs, dim=1)

        # Calculate loss
        # If criterion is CrossEntropyLoss, it expects logits.
        # We use the model outputs here to remain consistent
        # with the training setup you already used.
        loss = criterion(outputs, labels)

        # Add batch loss
        test_loss += loss.item() * labels.size(0)

        # Count correct predictions
        total_correct += (predicted == labels).sum().item()

        # Count total samples
        total_samples += labels.size(0)

        # Store predictions and actual labels
        all_predictions.extend(
            predicted.cpu().numpy()
        )

        all_true_labels.extend(
            labels.cpu().numpy()
        )


# ------------------------------------------------------------
# 4. Calculate final metrics
# ------------------------------------------------------------

average_test_loss = test_loss / total_samples

test_accuracy = (
    total_correct / total_samples
) * 100


# Convert lists to NumPy arrays
y_true = np.array(all_true_labels)
y_pred = np.array(all_predictions)


# ------------------------------------------------------------
# 5. Get class names
# ------------------------------------------------------------

class_names = [
    "actinic keratosis",
    "atypical melanocytic proliferation",
    "basal cell carcinoma",
    "benign",
    "melanoma",
    "nevus",
    "seborrheic keratosis",
    "squamous cell carcinoma"
]


# ------------------------------------------------------------
# 6. Print final results
# ------------------------------------------------------------

print("=" * 65)
print("FINAL TEST SET RESULTS")
print("=" * 65)

print()
print(
    f"Test Loss: {average_test_loss:.4f}"
)

print(
    f"Test Accuracy: {test_accuracy:.2f}%"
)


# ------------------------------------------------------------
# 7. Classification Report
# ------------------------------------------------------------

print()
print("CLASSIFICATION REPORT")
print("-" * 65)

print(
    classification_report(
        y_true,
        y_pred,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 8. Confusion Matrix
# ------------------------------------------------------------

print()
print("CONFUSION MATRIX")
print("-" * 65)

conf_matrix = confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(len(class_names))
)

print(conf_matrix)


# ------------------------------------------------------------
# 9. True Test Class Distribution
# ------------------------------------------------------------

print()
print("TRUE TEST CLASS DISTRIBUTION")
print("-" * 65)

unique_true, true_counts = np.unique(
    y_true,
    return_counts=True
)

for class_index, count in zip(
    unique_true,
    true_counts
):

    print(
        f"{class_index} -> "
        f"{class_names[class_index]}: "
        f"{count}"
    )


# ------------------------------------------------------------
# 10. Predicted Class Distribution
# ------------------------------------------------------------

print()
print("PREDICTED CLASS DISTRIBUTION")
print("-" * 65)

unique_pred, pred_counts = np.unique(
    y_pred,
    return_counts=True
)

for class_index, count in zip(
    unique_pred,
    pred_counts
):

    print(
        f"{class_index} -> "
        f"{class_names[class_index]}: "
        f"{count}"
    )


# ------------------------------------------------------------
# 11. Completion message
# ------------------------------------------------------------

print()
print("=" * 65)
print("TEST EVALUATION COMPLETED SUCCESSFULLY")
print("=" * 65)

In [ ]:
# ============================================================
# NEW CELL: RECREATE MISSING LABEL ENCODER
# ============================================================

from sklearn.preprocessing import LabelEncoder

# Create LabelEncoder
label_encoder = LabelEncoder()

# Fit using the exact diagnosis class names
label_encoder.fit([
    'actinic keratosis',
    'atypical melanocytic proliferation',
    'basal cell carcinoma',
    'benign',
    'melanoma',
    'nevus',
    'seborrheic keratosis',
    'squamous cell carcinoma'
])

print("============================================================")
print("LABEL ENCODER RECREATED SUCCESSFULLY")
print("============================================================")

print("\nClass Mapping:")

for i, class_name in enumerate(label_encoder.classes_):
    print(f"{i} -> {class_name}")

In [ ]:
# ============================================================
# CELL 1: BALANCED DATALOADERS + MILD CLASS WEIGHTS
# ============================================================

import torch
import torch.nn as nn
import numpy as np

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score

# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# ============================================================
# MODIFIED: NORMAL DATALOADERS
# Removed WeightedRandomSampler completely
# ============================================================

batch_size = 64

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("\n============================================================")
print("NORMAL DATALOADERS CREATED")
print("============================================================")

print("Batch size:", batch_size)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

# ============================================================
# TRAINING CLASS DISTRIBUTION
# ============================================================

class_counts = torch.bincount(y_train_tensor)

print("\nTraining Class Counts:")

for i, count in enumerate(class_counts):
    print(f"{i} -> {label_encoder.classes_[i]}: {count.item()}")

# ============================================================
# MODIFIED: CREATE MILD CLASS WEIGHTS
#
# We do NOT use the extremely large inverse-frequency weights.
#
# Step 1:
# Calculate inverse square-root frequency.
#
# Step 2:
# Normalize the weights.
#
# Step 3:
# Limit the maximum weight to avoid overcompensation.
# ============================================================

num_classes = len(class_counts)

class_weights = torch.sqrt(
    class_counts.sum().float() /
    (num_classes * class_counts.float())
)

# Normalize so average weight is approximately 1
class_weights = class_weights / class_weights.mean()

# MODIFIED: Limit maximum class weight
max_weight = 3.0

class_weights = torch.clamp(
    class_weights,
    min=0.5,
    max=max_weight
)

class_weights_tensor = class_weights.to(device)

print("\n============================================================")
print("MILD CLASS WEIGHTS")
print("============================================================")

for i, weight in enumerate(class_weights_tensor):
    print(
        f"{i} -> {label_encoder.classes_[i]}: "
        f"{weight.item():.4f}"
    )

print("\nClass weights tensor:")
print(class_weights_tensor)

# ============================================================
# CHECK DATA
# ============================================================

print("\n============================================================")
print("DATA PREPARATION COMPLETED SUCCESSFULLY")
print("============================================================")

print("✓ WeightedRandomSampler removed")
print("✓ Normal shuffled DataLoader used")
print("✓ Mild class weighting used")
print("✓ Extreme minority-class bias reduced")
print("✓ Ready for the new ANN model")

In [ ]:
# ============================================================
# CELL 2: CORRECTED BALANCED ANN MODEL
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

# ------------------------------------------------------------
# MODEL DEFINITION
# ------------------------------------------------------------

class BalancedANNModel(nn.Module):

    def __init__(self, input_size, num_classes):

        super(BalancedANNModel, self).__init__()

        # ----------------------------------------------------
        # HIDDEN LAYER 1
        # ----------------------------------------------------

        self.layer1 = nn.Linear(input_size, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.25)

        # ----------------------------------------------------
        # HIDDEN LAYER 2
        # ----------------------------------------------------

        self.layer2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.20)

        # ----------------------------------------------------
        # HIDDEN LAYER 3
        # ----------------------------------------------------

        self.layer3 = nn.Linear(64, 32)
        self.sigmoid = nn.Sigmoid()
        self.dropout3 = nn.Dropout(0.15)

        # ----------------------------------------------------
        # OUTPUT LAYER
        # ----------------------------------------------------

        self.output_layer = nn.Linear(32, num_classes)

        # ----------------------------------------------------
        # PROJECT REQUIREMENT:
        # Softmax is defined here.
        #
        # MODIFIED:
        # Softmax is NOT used inside forward() because
        # CrossEntropyLoss requires RAW LOGITS.
        # ----------------------------------------------------

        self.softmax = nn.Softmax(dim=1)


    def forward(self, x):

        # Layer 1
        x = self.layer1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout1(x)

        # Layer 2
        x = self.layer2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.dropout2(x)

        # Layer 3
        x = self.layer3(x)
        x = self.sigmoid(x)
        x = self.dropout3(x)

        # Output Layer
        # MODIFIED: Return RAW LOGITS
        x = self.output_layer(x)

        return x


# ============================================================
# CREATE MODEL
# ============================================================

input_size = X_train_tensor.shape[1]
num_classes = len(label_encoder.classes_)

model = BalancedANNModel(
    input_size=input_size,
    num_classes=num_classes
)

model = model.to(device)

# ============================================================
# LOSS FUNCTION
#
# MODIFIED:
# Uses the mild class weights created in Cell 1.
# ============================================================

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)

# ============================================================
# OPTIMIZER
# ============================================================

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0005,
    weight_decay=0.0001
)

# ============================================================
# LEARNING RATE SCHEDULER
#
# Reduces LR if validation performance stops improving.
# ============================================================

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=5
)

# ============================================================
# MODEL INFORMATION
# ============================================================

print("============================================================")
print("BALANCED ANN MODEL CREATED SUCCESSFULLY")
print("============================================================")

print("\nMODEL ARCHITECTURE")
print("------------------------------------------------------------")

print(model)

print("\nMODEL CONFIGURATION")
print("------------------------------------------------------------")

print("Input Features:", input_size)
print("Number of Classes:", num_classes)

print("\nNetwork Structure:")
print(f"{input_size} -> 128 -> 64 -> 32 -> {num_classes}")

print("\nPROJECT REQUIREMENTS")
print("------------------------------------------------------------")

print("✓ ReLU implemented")
print("✓ Sigmoid implemented")
print("✓ Softmax defined")
print("✓ Adam Optimizer implemented")

print("\nIMPORTANT MODIFICATIONS")
print("------------------------------------------------------------")

print("✓ Softmax removed from forward()")
print("✓ Model returns raw logits")
print("✓ Compatible with CrossEntropyLoss")
print("✓ Mild class weights used")
print("✓ No WeightedRandomSampler")
print("✓ Batch Normalization used")
print("✓ Dropout used")

print("\nTRAINING CONFIGURATION")
print("------------------------------------------------------------")

print("Loss Function: Mild Weighted CrossEntropyLoss")
print("Optimizer: Adam")
print("Learning Rate:", optimizer.param_groups[0]["lr"])
print("Weight Decay: 0.0001")

print("\n============================================================")
print("MODEL READY FOR BALANCED TRAINING")
print("============================================================")

In [ ]:
# ============================================================
# CELL 3: BALANCED MODEL TRAINING
# Macro F1 + Early Stopping + LR Scheduler
# ============================================================

import copy
import numpy as np
from sklearn.metrics import f1_score

# ============================================================
# TRAINING SETTINGS
# ============================================================

num_epochs = 150

# NEW: Early stopping patience
early_stopping_patience = 20

# Variables for tracking the best model
best_macro_f1 = -1.0
best_model_state = None

epochs_without_improvement = 0

# ============================================================
# TRAINING HISTORY
# ============================================================

train_losses = []
train_accuracies = []

val_losses = []
val_accuracies = []
val_macro_f1_scores = []

# ============================================================
# TRAINING START
# ============================================================

print("============================================================")
print("BALANCED TRAINING STARTED")
print("============================================================")

for epoch in range(num_epochs):

    # --------------------------------------------------------
    # TRAINING MODE
    # --------------------------------------------------------

    model.train()

    running_train_loss = 0.0
    correct_train = 0
    total_train = 0

    # --------------------------------------------------------
    # TRAINING LOOP
    # --------------------------------------------------------

    for batch_features, batch_labels in train_loader:

        # Move data to device
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        # Reset gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(batch_features)

        # Calculate loss
        loss = criterion(
            outputs,
            batch_labels
        )

        # Backpropagation
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        # Update model
        optimizer.step()

        # Store loss
        running_train_loss += (
            loss.item() * batch_features.size(0)
        )

        # Predictions
        _, predicted = torch.max(
            outputs,
            dim=1
        )

        # Calculate correct predictions
        correct_train += (
            predicted == batch_labels
        ).sum().item()

        total_train += batch_labels.size(0)

    # --------------------------------------------------------
    # TRAINING METRICS
    # --------------------------------------------------------

    epoch_train_loss = (
        running_train_loss / total_train
    )

    epoch_train_accuracy = (
        100 * correct_train / total_train
    )

    train_losses.append(
        epoch_train_loss
    )

    train_accuracies.append(
        epoch_train_accuracy
    )

    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    running_val_loss = 0.0
    correct_val = 0
    total_val = 0

    all_val_predictions = []
    all_val_labels = []

    # Disable gradient calculation
    with torch.no_grad():

        for batch_features, batch_labels in val_loader:

            # Move data to device
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            # Forward pass
            outputs = model(batch_features)

            # Validation loss
            loss = criterion(
                outputs,
                batch_labels
            )

            running_val_loss += (
                loss.item() * batch_features.size(0)
            )

            # Predictions
            _, predicted = torch.max(
                outputs,
                dim=1
            )

            correct_val += (
                predicted == batch_labels
            ).sum().item()

            total_val += batch_labels.size(0)

            # Store predictions for Macro F1
            all_val_predictions.extend(
                predicted.cpu().numpy()
            )

            all_val_labels.extend(
                batch_labels.cpu().numpy()
            )

    # --------------------------------------------------------
    # VALIDATION METRICS
    # --------------------------------------------------------

    epoch_val_loss = (
        running_val_loss / total_val
    )

    epoch_val_accuracy = (
        100 * correct_val / total_val
    )

    # NEW: Macro F1 Score
    epoch_val_macro_f1 = f1_score(
        all_val_labels,
        all_val_predictions,
        average="macro",
        zero_division=0
    )

    val_losses.append(
        epoch_val_loss
    )

    val_accuracies.append(
        epoch_val_accuracy
    )

    val_macro_f1_scores.append(
        epoch_val_macro_f1
    )

    # ========================================================
    # LEARNING RATE SCHEDULER
    # MODIFIED:
    # Scheduler now monitors Macro F1
    # ========================================================

    scheduler.step(
        epoch_val_macro_f1
    )

    current_lr = optimizer.param_groups[0]["lr"]

    # ========================================================
    # PRINT EPOCH RESULTS
    # ========================================================

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Train Acc: {epoch_train_accuracy:.2f}% | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {epoch_val_accuracy:.2f}% | "
        f"Val Macro F1: {epoch_val_macro_f1:.4f} | "
        f"LR: {current_lr:.6f}"
    )

    # ========================================================
    # SAVE BEST MODEL BASED ON MACRO F1
    # ========================================================

    if epoch_val_macro_f1 > best_macro_f1:

        best_macro_f1 = epoch_val_macro_f1

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        epochs_without_improvement = 0

    else:

        epochs_without_improvement += 1

    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if (
        epochs_without_improvement
        >= early_stopping_patience
    ):

        print("\n============================================================")
        print(
            f"EARLY STOPPING ACTIVATED "
            f"AFTER {epoch + 1} EPOCHS"
        )

        print(
            f"Best Validation Macro F1: "
            f"{best_macro_f1:.4f}"
        )

        print("============================================================")

        break


# ============================================================
# RESTORE BEST MODEL
# ============================================================

if best_model_state is not None:

    model.load_state_dict(
        best_model_state
    )


# ============================================================
# FINAL TRAINING SUMMARY
# ============================================================

print("\n============================================================")
print("BALANCED TRAINING COMPLETED SUCCESSFULLY")
print("============================================================")

print(
    f"Best Validation Macro F1: "
    f"{best_macro_f1:.4f}"
)

print(
    f"Total Epochs Completed: "
    f"{len(train_losses)}"
)

print("\n✓ Best model restored successfully")
print("✓ Macro F1 used for model selection")
print("✓ Validation accuracy tracked")
print("✓ Learning rate scheduler used")
print("✓ Gradient clipping used")
print("✓ Early stopping used")

print("\n============================================================")
print("READY FOR FINAL TEST EVALUATION")
print("============================================================")

In [ ]:
# ============================================================
# RECREATE THE BALANCED MODEL
# ============================================================

import torch
import torch.nn as nn

class BalancedANNModel(nn.Module):
    
    def __init__(self, input_size, num_classes):
        super(BalancedANNModel, self).__init__()

        self.layer1 = nn.Linear(input_size, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.25)

        self.layer2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.20)

        self.layer3 = nn.Linear(64, 32)
        self.sigmoid = nn.Sigmoid()
        self.dropout3 = nn.Dropout(0.15)

        self.output_layer = nn.Linear(32, num_classes)

        # Defined for project requirement
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):

        x = self.layer1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout1(x)

        x = self.layer2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.dropout2(x)

        x = self.layer3(x)
        x = self.sigmoid(x)
        x = self.dropout3(x)

        # Return RAW LOGITS
        x = self.output_layer(x)

        return x


# ============================================================
# CREATE MODEL
# ============================================================

input_size = 82
num_classes = 8

model = BalancedANNModel(
    input_size=input_size,
    num_classes=num_classes
)

print("Model recreated successfully!")

print("\nMODEL ARCHITECTURE")
print("=" * 60)
print(model)

In [ ]:
torch.save(model.state_dict(), "best_balanced_model.pth")

In [ ]:
# Load the trained weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.load_state_dict(
    torch.load(
        "best_balanced_model.pth",
        map_location=device
    )
)

model = model.to(device)

print("Best trained model loaded successfully!")

In [ ]:
# ============================================================
# COMPLETE TEST DATA RECOVERY
# ============================================================

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

# ------------------------------------------------------------
# CHECK AVAILABLE TEST VARIABLES
# ------------------------------------------------------------

print("=" * 65)
print("CHECKING AVAILABLE VARIABLES")
print("=" * 65)

variables_to_check = [
    "X_test_encoded",
    "y_test_encoded",
    "X_test_processed",
    "X_test",
    "y_test"
]

for variable_name in variables_to_check:
    if variable_name in globals():
        value = globals()[variable_name]
        print(f"✓ {variable_name} exists | Shape: {np.array(value).shape}")
    else:
        print(f"✗ {variable_name} not found")


# ------------------------------------------------------------
# RECREATE TEST FEATURE TENSOR
# ------------------------------------------------------------

if "X_test_encoded" in globals():

    X_test_array = np.asarray(
        X_test_encoded,
        dtype=np.float32
    )

elif "X_test_processed" in globals():

    X_test_array = np.asarray(
        X_test_processed,
        dtype=np.float32
    )

else:

    raise NameError(
        "\nTest feature data was not found.\n"
        "You need to rerun the preprocessing cell that created "
        "X_test_encoded."
    )


# ------------------------------------------------------------
# RECREATE TEST LABEL TENSOR
# ------------------------------------------------------------

if "y_test_encoded" in globals():

    y_test_array = np.asarray(
        y_test_encoded,
        dtype=np.int64
    )

elif "y_test" in globals():

    # Use only if y_test already contains encoded integer labels
    y_test_array = np.asarray(
        y_test,
        dtype=np.int64
    )

else:

    raise NameError(
        "\nTest labels were not found.\n"
        "You need to rerun the target encoding cell that created "
        "y_test_encoded."
    )


# ------------------------------------------------------------
# CREATE PYTORCH TENSORS
# ------------------------------------------------------------

X_test_tensor = torch.tensor(
    X_test_array,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test_array,
    dtype=torch.long
)


# ------------------------------------------------------------
# CREATE DATASET
# ------------------------------------------------------------

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)


# ------------------------------------------------------------
# CREATE DATALOADER
# ------------------------------------------------------------

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)


# ------------------------------------------------------------
# FINAL VERIFICATION
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("TEST DATA RECOVERY COMPLETED SUCCESSFULLY")
print("=" * 65)

print(f"X_test_tensor shape: {X_test_tensor.shape}")
print(f"y_test_tensor shape: {y_test_tensor.shape}")
print(f"Number of test samples: {len(test_dataset)}")
print(f"Number of batches: {len(test_loader)}")
print(f"Batch size: 64")

print("\n✓ X_test_tensor recreated")
print("✓ y_test_tensor recreated")
print("✓ test_dataset recreated")
print("✓ test_loader recreated")
print("=" * 65)

In [ ]:
# ============================================================
# RECREATE TEST DATALOADER
# ============================================================

import torch
from torch.utils.data import TensorDataset, DataLoader

# ------------------------------------------------------------
# Make sure X_test_tensor and y_test_tensor exist
# ------------------------------------------------------------

print("Checking tensors...")

print("X_test_tensor shape:", X_test_tensor.shape)
print("y_test_tensor shape:", y_test_tensor.shape)


# ------------------------------------------------------------
# Create Test Dataset
# ------------------------------------------------------------

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)


# ------------------------------------------------------------
# Create Test DataLoader
# ------------------------------------------------------------

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)


print("\n" + "=" * 60)
print("TEST DATALOADER CREATED SUCCESSFULLY")
print("=" * 60)

print("Testing samples:", len(test_dataset))
print("Batch size: 64")
print("Number of batches:", len(test_loader))

In [ ]:
# ============================================================
# FINAL TEST EVALUATION
# ============================================================

import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

print("=" * 65)
print("FINAL TEST SET EVALUATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. Evaluation mode
# ------------------------------------------------------------

model.eval()

test_losses = []
all_predictions = []
all_true_labels = []

# ------------------------------------------------------------
# 2. Run prediction on test set
# ------------------------------------------------------------

with torch.no_grad():

    for features, labels in test_loader:

        # Move data to device
        features = features.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(features)

        # Calculate loss
        loss = criterion(outputs, labels)

        test_losses.append(loss.item())

        # Get predicted class
        _, predictions = torch.max(outputs, 1)

        # Store predictions and true labels
        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_true_labels.extend(
            labels.cpu().numpy()
        )


# ------------------------------------------------------------
# 3. Convert results to NumPy arrays
# ------------------------------------------------------------

all_predictions = np.array(all_predictions)
all_true_labels = np.array(all_true_labels)

# ------------------------------------------------------------
# 4. Calculate metrics
# ------------------------------------------------------------

test_loss = np.mean(test_losses)

test_accuracy = accuracy_score(
    all_true_labels,
    all_predictions
)

test_macro_f1 = f1_score(
    all_true_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

test_weighted_f1 = f1_score(
    all_true_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)

# ------------------------------------------------------------
# 5. Class names
# ------------------------------------------------------------

class_names = [
    "actinic keratosis",
    "atypical melanocytic proliferation",
    "basal cell carcinoma",
    "benign",
    "melanoma",
    "nevus",
    "seborrheic keratosis",
    "squamous cell carcinoma"
]

# ------------------------------------------------------------
# 6. Print final results
# ------------------------------------------------------------

print()
print("=" * 65)
print("FINAL TEST SET RESULTS")
print("=" * 65)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Test Macro F1 Score: {test_macro_f1:.4f}")
print(f"Test Weighted F1 Score: {test_weighted_f1:.4f}")

# ------------------------------------------------------------
# 7. Classification report
# ------------------------------------------------------------

print("\nCLASSIFICATION REPORT")
print("-" * 65)

print(
    classification_report(
        all_true_labels,
        all_predictions,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        zero_division=0
    )
)

# ------------------------------------------------------------
# 8. Confusion matrix
# ------------------------------------------------------------

conf_matrix = confusion_matrix(
    all_true_labels,
    all_predictions,
    labels=np.arange(len(class_names))
)

print("\nCONFUSION MATRIX")
print("-" * 65)

print(conf_matrix)

# ------------------------------------------------------------
# 9. True test class distribution
# ------------------------------------------------------------

print("\nTRUE TEST CLASS DISTRIBUTION")
print("-" * 65)

unique_true, true_counts = np.unique(
    all_true_labels,
    return_counts=True
)

for class_index, count in zip(unique_true, true_counts):

    print(
        f"{class_index} -> "
        f"{class_names[class_index]}: "
        f"{count}"
    )

# ------------------------------------------------------------
# 10. Predicted class distribution
# ------------------------------------------------------------

print("\nPREDICTED CLASS DISTRIBUTION")
print("-" * 65)

unique_predicted, predicted_counts = np.unique(
    all_predictions,
    return_counts=True
)

for class_index, count in zip(
    unique_predicted,
    predicted_counts
):

    print(
        f"{class_index} -> "
        f"{class_names[class_index]}: "
        f"{count}"
    )

print()
print("=" * 65)
print("TEST EVALUATION COMPLETED SUCCESSFULLY")
print("=" * 65)

In [ ]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Convert features to tensors
X_train_tensor = torch.tensor(
    np.asarray(X_train_encoded, dtype=np.float32),
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    np.asarray(X_val_encoded, dtype=np.float32),
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    np.asarray(X_test_encoded, dtype=np.float32),
    dtype=torch.float32
)

# Convert labels to tensors
y_train_tensor = torch.tensor(
    np.asarray(y_train_encoded, dtype=np.int64),
    dtype=torch.long
)

y_val_tensor = torch.tensor(
    np.asarray(y_val_encoded, dtype=np.int64),
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    np.asarray(y_test_encoded, dtype=np.int64),
    dtype=torch.long
)

# Create datasets
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

# Create DataLoaders
batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("=" * 60)
print("ALL PYTORCH DATA RECREATED SUCCESSFULLY")
print("=" * 60)

print("Training:", X_train_tensor.shape)
print("Validation:", X_val_tensor.shape)
print("Testing:", X_test_tensor.shape)

print("\nTrain Loader:", len(train_loader), "batches")
print("Validation Loader:", len(val_loader), "batches")
print("Test Loader:", len(test_loader), "batches")

print("\nUsing device:", device)

In [ ]:
# ============================================================
# RECREATE TRAIN / VALIDATION / TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
import numpy as np

# ------------------------------------------------------------
# Separate features and target
# ------------------------------------------------------------

X = improved_feature_dataset.drop("Diagnosis", axis=1)
y = improved_feature_dataset["Diagnosis"]

# ------------------------------------------------------------
# First split: Train = 64%, Temporary = 36%
# ------------------------------------------------------------

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.36,
    random_state=42,
    stratify=y
)

# ------------------------------------------------------------
# Second split:
# Validation = 16%
# Test = 20%
# ------------------------------------------------------------

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5556,
    random_state=42,
    stratify=y_temp
)

print("=" * 60)
print("SPLIT COMPLETED")
print("=" * 60)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

In [1]:
###################################################################################################

In [ ]:
improved_feature_dataset